In [9]:
# ============================================================
# DAY 8 — FULL UNIVERSE EXPANSION
# AI Stock Screener — Indian Markets
# Stage 1: yfinance pre-filters on full 4,959 stock universe
# ============================================================

# ── CELL 1: IMPORTS & SETUP ──────────────────────────────────
import pandas as pd
import numpy as np
import yfinance as yf
import pickle
import os
import time
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data', exist_ok=True)

print("✅ Imports done")
print(f"yfinance version: {yf.__version__}")

✅ Imports done
yfinance version: 1.2.0


In [10]:
# ── CELL 2: LOAD MASTER STOCKS ───────────────────────────────
master = pd.read_csv('../data/master_stocks.csv')

print(f"Total stocks in universe : {len(master)}")
print(f"NSE stocks               : {len(master[master['Exchange'] == 'NSE'])}")
print(f"BSE stocks               : {len(master[master['Exchange'] == 'BSE'])}")
print(f"\nSample:")
print(master.head(5).to_string(index=False))

Total stocks in universe : 4959
NSE stocks               : 2117
BSE stocks               : 2842

Sample:
    Symbol                             Company Name Exchange
 20MICRONS                       20 Microns Limited      NSE
21STCENMGM 21st Century Management Services Limited      NSE
    360ONE                      360 ONE WAM LIMITED      NSE
 3IINFOLTD                      3i Infotech Limited      NSE
   3MINDIA                         3M India Limited      NSE


In [11]:
# ── CELL 3: REMOVE COMPANY NAME DUPLICATES ───────────────────
# Symbol dedup already done in Day 1
# But same company can appear as NSE+BSE with different symbols
# Strategy: if company name matches (fuzzy), keep NSE, drop BSE

nse_stocks = master[master['Exchange'] == 'NSE'].copy()
bse_stocks = master[master['Exchange'] == 'BSE'].copy()

# Normalize company names for matching
def normalize_name(name):
    name = str(name).upper().strip()
    # Remove common suffixes
    for suffix in [' LIMITED', ' LTD', ' LTD.', ' PRIVATE', 
                   ' PVT', ' PVT.', ' INC', ' CORP', ' CO.']:
        name = name.replace(suffix, '')
    # Remove special characters
    name = ''.join(e for e in name if e.isalnum() or e == ' ')
    return name.strip()

nse_stocks['Name_Norm'] = nse_stocks['Company Name'].apply(normalize_name)
bse_stocks['Name_Norm'] = bse_stocks['Company Name'].apply(normalize_name)

# Find BSE stocks whose normalized name exists in NSE
nse_names = set(nse_stocks['Name_Norm'])
bse_duplicates = bse_stocks[bse_stocks['Name_Norm'].isin(nse_names)]
bse_unique     = bse_stocks[~bse_stocks['Name_Norm'].isin(nse_names)]

print(f"NSE stocks              : {len(nse_stocks)}")
print(f"BSE stocks (original)   : {len(bse_stocks)}")
print(f"BSE duplicates (by name): {len(bse_duplicates)}")
print(f"BSE unique (keep)       : {len(bse_unique)}")

# Final clean universe
master_clean = pd.concat([nse_stocks, bse_unique], ignore_index=True)
master_clean = master_clean.drop(columns=['Name_Norm'])

print(f"\nFinal clean universe    : {len(master_clean)}")
print(f"  NSE : {len(master_clean[master_clean['Exchange'] == 'NSE'])}")
print(f"  BSE : {len(master_clean[master_clean['Exchange'] == 'BSE'])}")

# Show sample duplicates caught
print(f"\nSample BSE duplicates removed:")
print(bse_duplicates[['Symbol', 'Company Name']].head(10).to_string(index=False))

NSE stocks              : 2117
BSE stocks (original)   : 2842
BSE duplicates (by name): 6
BSE unique (keep)       : 2836

Final clean universe    : 4953
  NSE : 2117
  BSE : 2836

Sample BSE duplicates removed:
   Symbol                          Company Name
   SINGER                  Singer India Limited
     GVTD          GE Vernova T&D India Limited
SOLARAPP1 SOLARA ACTIVE PHARMA SCIENCES LIMITED
   SSFLPP   Spandana Sphoorty Financial Limited
    ATLPP            Allcargo Terminals Limited
  KRISHPP               KRISHIVAL FOODS LIMITED


In [12]:
# ── CELL 4: BUILD YFINANCE TICKER SYMBOLS ────────────────────

def get_ticker(symbol, exchange):
    if exchange == 'NSE':
        return f"{symbol}.NS"
    else:
        return f"{symbol}.BO"

master_clean['Ticker'] = master_clean.apply(
    lambda row: get_ticker(row['Symbol'], row['Exchange']), axis=1
)

print(f"Total tickers: {len(master_clean)}")
print(f"\nSample NSE tickers:")
print(master_clean[master_clean['Exchange']=='NSE']['Ticker'].head(5).tolist())
print(f"\nSample BSE tickers:")
print(master_clean[master_clean['Exchange']=='BSE']['Ticker'].head(5).tolist())

Total tickers: 4953

Sample NSE tickers:
['20MICRONS.NS', '21STCENMGM.NS', '360ONE.NS', '3IINFOLTD.NS', '3MINDIA.NS']

Sample BSE tickers:
['AMBALALSA.BO', 'ANDHRAPET.BO', 'ANSALAPI.BO', 'UTIQUE.BO', 'ARUNAHTEL.BO']


In [16]:
# ── CELL 5: PRE-FILTER ─────────────────────

from datetime import datetime

MIN_DAILY_VALUE  = 10_00_000
MIN_MARKET_CAP   = 50_00_00_000
MIN_PRICE        = 5
MIN_LISTING_DAYS = 365
MIN_DATA_ROWS    = 200

def flatten_df(df):
    """Flatten MultiIndex columns from yfinance 1.2.0"""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]
    return df

def check_stock(symbol, exchange, ticker):
    result = {
        'Symbol'           : symbol,
        'Exchange'         : exchange,
        'Ticker'           : ticker,
        'passed'           : False,
        'fail_reason'      : None,
        'Price'            : None,
        'Avg_Daily_Value_L': None,
        'Market_Cap_Cr'    : None,
        'Data_Rows'        : None,
    }
    try:
        df_60 = yf.download(ticker, period='60d', interval='1d',
                            progress=False, auto_adjust=True)
        df_60 = flatten_df(df_60)

        if df_60 is None or len(df_60) < 5:
            result['fail_reason'] = 'No data'
            return result

        df_full = yf.download(ticker, period='max', interval='1d',
                              progress=False, auto_adjust=True)
        df_full = flatten_df(df_full)

        current_price = float(df_60['Close'].iloc[-1])
        avg_value     = float(df_60['Volume'].mean()) * current_price

        result['Price']             = round(current_price, 2)
        result['Avg_Daily_Value_L'] = round(avg_value / 1e5, 2)
        result['Data_Rows']         = len(df_full)

        # Filter checks
        if current_price < MIN_PRICE:
            result['fail_reason'] = f'Price Rs {current_price:.1f} < Rs 5'
            return result

        if avg_value < MIN_DAILY_VALUE:
            result['fail_reason'] = f'Daily value Rs {avg_value/1e5:.1f}L < Rs 10L'
            return result

        market_cap = None
        try:
            info       = yf.Ticker(ticker).fast_info
            market_cap = getattr(info, 'market_cap', None)
        except:
            pass

        if market_cap is not None:
            result['Market_Cap_Cr'] = round(market_cap / 1e7, 2)
            if market_cap < MIN_MARKET_CAP:
                result['fail_reason'] = f'MCap Rs {market_cap/1e7:.1f}Cr < Rs 50Cr'
                return result

        if len(df_full) > 0:
            first_date   = df_full.index[0]
            listing_days = (datetime.now() - first_date.to_pydatetime().replace(tzinfo=None)).days
            if listing_days < MIN_LISTING_DAYS:
                result['fail_reason'] = f'Listed only {listing_days} days'
                return result

        if len(df_full) < MIN_DATA_ROWS:
            result['fail_reason'] = f'Only {len(df_full)} rows'
            return result

        result['passed'] = True
        return result

    except Exception as e:
        result['fail_reason'] = f'Error: {str(e)[:60]}'
        return result

print("✅ Filter definitions ready")

✅ Filter definitions ready


In [18]:
# ── CELL 6: FULL UNIVERSE PRE-FILTER ─────────────────────────
# Set to True ONLY when you want to force a fresh quarterly run
FORCE_RERUN = False

if not FORCE_RERUN and os.path.exists('../data/prefilt_passed.csv'):
    existing = pd.read_csv('../data/prefilt_passed.csv')
    print(f"⚠️  Pre-filter already completed — {len(existing)} stocks")
    print(f"    Set FORCE_RERUN = True to re-run quarterly refresh")

else:
    CHECKPOINT_FILE = '../data/prefilt_checkpoint.pkl'
    OUTPUT_FILE     = '../data/prefilt_results.csv'

    def is_checkpoint_valid(checkpoint):
        results = checkpoint.get('results', [])
        if len(results) < 50:
            return True
        first_50      = results[:50]
        first_50_pass = sum(1 for r in first_50 if r.get('passed', False))
        pass_rate     = first_50_pass / 50
        if pass_rate == 0:
            print(f"  ⚠️  Checkpoint invalid — 0% pass rate, starting fresh")
            return False
        has_prices = any(r.get('Price') is not None for r in first_50)
        if not has_prices:
            print(f"  ⚠️  Checkpoint invalid — no price data, starting fresh")
            return False
        print(f"  ✅ Checkpoint valid — first 50 pass rate: {pass_rate:.0%}")
        return True

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'rb') as f:
            checkpoint = pickle.load(f)
        if is_checkpoint_valid(checkpoint):
            results   = checkpoint['results']
            start_idx = checkpoint['next_idx']
            print(f"Resuming from stock {start_idx}/{len(master_clean)}")
        else:
            os.remove(CHECKPOINT_FILE)
            results   = []
            start_idx = 0
            print(f"Starting fresh — {len(master_clean)} stocks")
    else:
        results   = []
        start_idx = 0
        print(f"Starting fresh — {len(master_clean)} stocks")

    total  = len(master_clean)
    passed = sum(1 for r in results if r.get('passed', False))
    failed = len(results) - passed

    print(f"Already processed: {len(results)} | Passed: {passed} | Failed: {failed}")
    print("-" * 60)

    for i in range(start_idx, total):
        row    = master_clean.iloc[i]
        symbol = row['Symbol']
        exch   = row['Exchange']
        ticker = row['Ticker']

        r = check_stock(symbol, exch, ticker)
        results.append(r)

        if r['passed']:
            passed += 1
        else:
            failed += 1

        if (i + 1) % 50 == 0 or (i + 1) == total:
            pct = (i + 1) / total * 100
            print(f"  [{i+1:4d}/{total}] {pct:5.1f}% | Passed: {passed} | Failed: {failed}")

        if (i + 1) % 100 == 0:
            with open(CHECKPOINT_FILE, 'wb') as f:
                pickle.dump({
                    'results'   : results,
                    'next_idx'  : i + 1,
                    'version'   : '2.0',
                    'created_at': datetime.now().strftime('%Y-%m-%d %H:%M'),
                }, f)

        time.sleep(0.5)

    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_FILE, index=False)

    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

    passed_df = results_df[results_df['passed'] == True]

    print(f"\n✅ Pre-filter complete")
    print(f"   Total processed : {len(results_df)}")
    print(f"   Passed          : {len(passed_df)}")
    print(f"   Failed          : {len(results_df) - len(passed_df)}")
    print(f"\nTop fail reasons:")
    print(results_df[results_df['passed'] == False]['fail_reason']
          .str.split('<').str[0].str.strip()
          .value_counts().head(10).to_string())

    passed_df.to_csv('../data/prefilt_passed.csv', index=False)
    print(f"\n✅ Passed stocks saved: data/prefilt_passed.csv")

⚠️  Pre-filter already completed — 2142 stocks
    Set FORCE_RERUN = True to re-run quarterly refresh


In [19]:
# ── CELL 7: ANALYSE PRE-FILTER RESULTS ───────────────────────
passed_df = pd.read_csv('../data/prefilt_passed.csv')

print(f"Total passed : {len(passed_df)}")
print(f"\nExchange breakdown:")
print(passed_df['Exchange'].value_counts().to_string())

print(f"\nMarket cap distribution:")
passed_df['Cap_Category'] = pd.cut(
    passed_df['Market_Cap_Cr'],
    bins=[0, 1000, 5000, 20000, float('inf')],
    labels=['Small Cap', 'Mid Cap', 'Mini Large', 'Large Cap']
)
print(passed_df['Cap_Category'].value_counts().to_string())

print(f"\nDaily value distribution (Lakhs):")
bins   = [0, 10, 50, 100, 500, 1000, float('inf')]
labels = ['<10L', '10-50L', '50-100L', '100-500L', '500L-1Cr', '>1Cr']
passed_df['Val_Bucket'] = pd.cut(passed_df['Avg_Daily_Value_L'], bins=bins, labels=labels)
print(passed_df['Val_Bucket'].value_counts().sort_index().to_string())

print(f"\nSample passed stocks:")
print(passed_df[['Symbol', 'Exchange', 'Price', 'Avg_Daily_Value_L', 'Market_Cap_Cr']].head(10).to_string(index=False))

Total passed : 2142

Exchange breakdown:
Exchange
NSE    1692
BSE     450

Market cap distribution:
Cap_Category
Small Cap     881
Mid Cap       578
Mini Large    366
Large Cap     306

Daily value distribution (Lakhs):
Val_Bucket
<10L          0
10-50L      591
50-100L     186
100-500L    435
500L-1Cr    206
>1Cr        724

Sample passed stocks:
    Symbol Exchange    Price  Avg_Daily_Value_L  Market_Cap_Cr
 20MICRONS      NSE   151.02             495.99         534.54
    360ONE      NSE  1040.60           10101.78       42239.61
 3IINFOLTD      NSE    13.82              62.49         287.09
   3MINDIA      NSE 32270.00            1477.32       36352.38
    5PAISA      NSE   291.60             627.78         911.30
   63MOONS      NSE   531.45             916.80        2448.99
   AAATECH      NSE    93.00              49.10         119.29
 AADHARHFC      NSE   449.85            1868.57       19561.12
AAREYDRUGS      NSE    69.28             169.96         196.47
     AARON      NSE 

In [24]:
passed_df = pd.read_csv('../data/prefilt_passed.csv')
print(f"Stocks to scrape: {len(passed_df)}")

Stocks to scrape: 2142


In [38]:
# ── CELL 8: FULL UNIVERSE FUNDAMENTAL SCRAPER ────────────────
# Handles NSE symbols + BSE numeric code fallback automatically
# Checkpointing, resume, FORCE_RESCRAPE guard

from bs4 import BeautifulSoup
import requests

FUND_CHECKPOINT = '../data/fund_scrape_checkpoint.pkl'
FUND_RAW_FILE   = '../data/raw_stock_data_full.pkl'
FUND_OUTPUT     = '../data/fundamental_metrics_full.csv'
FORCE_RESCRAPE  = False

# ── GUARD ────────────────────────────────────────────────────
if not FORCE_RESCRAPE and os.path.exists(FUND_OUTPUT):
    existing = pd.read_csv(FUND_OUTPUT)
    print(f"⚠️  Fundamental scrape already completed — {len(existing)} stocks")
    print(f"    Set FORCE_RESCRAPE = True to re-run quarterly refresh")

else:
    # ── BSE CODE MAP ─────────────────────────────────────────
    bse_raw = pd.read_csv('../data/bse_stocks.csv',
                          index_col=False,
                          usecols=['Security Code', 'Security Id',
                                   'Issuer Name', 'Status',
                                   'Group', 'Face Value',
                                   'ISIN No', 'Instrument'])
    bse_raw['Security Id'] = bse_raw['Security Id'].str.strip()
    bse_code_map = dict(zip(bse_raw['Security Id'],
                            bse_raw['Security Code']))
    print(f"✅ BSE code map loaded: {len(bse_code_map)} symbols")

    # ── SCRAPER FUNCTIONS ────────────────────────────────────
    def extract_table(table):
        try:
            thead   = table.find('thead')
            columns = []
            if thead:
                for th in thead.find_all('th'):
                    text = th.get_text(strip=True)
                    columns.append(text if text else 'Metric')
            tbody = table.find('tbody')
            if not tbody:
                return None
            rows = tbody.find_all('tr')
            data = {}
            for row in rows:
                cells = row.find_all('td')
                if not cells:
                    continue
                metric_name = cells[0].get_text(strip=True)
                metric_name = metric_name.replace('+', '').strip()
                skip_keywords = ['Raw PDF', 'PDF', 'Source']
                if any(kw in metric_name for kw in skip_keywords):
                    continue
                values = []
                for cell in cells[1:]:
                    val = cell.get_text(strip=True)
                    val = val.replace(',', '').replace('%', '').strip()
                    try:
                        values.append(float(val))
                    except:
                        values.append(val if val else None)
                data[metric_name] = values
            if not data:
                return None
            col_names = columns[1:] if len(columns) > 1 else list(range(len(values)))
            return pd.DataFrame(data, index=col_names).T
        except:
            return None

    def scrape_screener(lookup):
        """
        Scrape Screener.in for a given lookup (NSE symbol or BSE code).
        Tries consolidated first, falls back to standalone.
        """
        try:
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
            urls = [
                f"https://www.screener.in/company/{lookup}/consolidated/",
                f"https://www.screener.in/company/{lookup}/",
            ]
            soup = None
            for url in urls:
                response = requests.get(url, headers=headers, timeout=15)
                if response.status_code != 200:
                    continue
                temp_soup = BeautifulSoup(response.text, 'html.parser')
                # Check if P&L has real data
                has_data = False
                for section in temp_soup.find_all('section'):
                    heading = section.find('h2')
                    if heading and 'Profit' in heading.text:
                        table = section.find('table')
                        if table:
                            thead = table.find('thead')
                            if thead and len(thead.find_all('th')) > 1:
                                has_data = True
                        break
                if has_data:
                    soup = temp_soup
                    break
            if soup is None:
                return None

            target_sections = [
                'Quarterly Results', 'Profit & Loss', 'Balance Sheet',
                'Cash Flows', 'Ratios', 'Shareholding Pattern',
            ]
            result = {}
            for section in soup.find_all('section'):
                heading = section.find('h2')
                if not heading:
                    continue
                section_name = heading.text.strip()
                if section_name not in target_sections:
                    continue
                table = section.find('table')
                if not table:
                    continue
                df = extract_table(table)
                if df is not None:
                    result[section_name] = df
            for section in soup.find_all('section'):
                heading = section.find('h2')
                if heading and 'Peer' in heading.text:
                    links       = section.find_all('a')
                    sector_info = {}
                    titles      = ['Broad Sector', 'Sector',
                                   'Broad Industry', 'Industry']
                    for link in links:
                        title = link.get('title', '')
                        if title in titles:
                            sector_info[title] = link.text.strip()
                    result['Sector Info'] = sector_info
                    break
            return result if result else None
        except:
            return None

    def scrape_stock(symbol, exchange):
        """
        Smart scraper — tries NSE symbol first.
        If fails and BSE stock, retries with BSE numeric code.
        """
        # Try NSE symbol first (works for all NSE + many BSE)
        result = scrape_screener(symbol)
        if result:
            return result

        # If BSE stock and symbol lookup failed → try BSE numeric code
        if exchange == 'BSE':
            bse_code = bse_code_map.get(symbol)
            if bse_code:
                result = scrape_screener(str(int(bse_code)))
                if result:
                    return result

        return None

    def safe_cagr(end_val, start_val, years):
        try:
            if end_val and start_val and start_val > 0 and end_val > 0 and years > 0:
                return round(((end_val / start_val) ** (1/years) - 1) * 100, 2)
        except:
            pass
        return None

    def extract_all_metrics(symbol, data):
        m = {'Symbol': symbol}
        try:
            sector_info       = data.get('Sector Info', {})
            m['Sector']       = sector_info.get('Sector', None)
            m['Industry']     = sector_info.get('Industry', None)
            m['Broad Sector'] = sector_info.get('Broad Sector', None)

            financial_keywords = ['Financial', 'Banking', 'Insurance',
                                   'NBFC', 'Lending', 'Microfinance']
            is_financial = any(kw in str(m.get('Sector', ''))
                               for kw in financial_keywords)

            if is_financial:
                revenue_row = 'Revenue'
                margin_row  = 'Financing Margin %'
                debt_row    = 'Borrowing'
            else:
                revenue_row = 'Sales'
                margin_row  = 'OPM %'
                debt_row    = 'Borrowings'

            pl = data.get('Profit & Loss')
            if pl is not None:
                pl_clean = pl.drop(columns=['TTM'], errors='ignore')
                if revenue_row in pl_clean.index:
                    sales = pl_clean.loc[revenue_row].dropna()
                    if len(sales) >= 6:
                        m['Revenue CAGR 5Y'] = safe_cagr(
                            sales.iloc[-1], sales.iloc[-6], 5)
                    if len(sales) >= 11:
                        m['Revenue CAGR 10Y'] = safe_cagr(
                            sales.iloc[-1], sales.iloc[-11], 10)
                    if len(sales) >= 3:
                        yrs = min(len(sales) - 1, 10)
                        base = sales.iloc[-yrs - 1]
                        if base > 0:
                            m['Revenue CAGR Max']  = safe_cagr(
                                sales.iloc[-1], base, yrs)
                            m['Revenue CAGR Years'] = yrs

                if margin_row in pl_clean.index:
                    opm = pl_clean.loc[margin_row].dropna()
                    m['Avg OPM 5Y']  = round(opm.iloc[-5:].mean(), 2) if len(opm) >= 5 else (round(opm.mean(), 2) if len(opm) > 0 else None)
                    m['Avg OPM 10Y'] = round(opm.iloc[-10:].mean(), 2) if len(opm) >= 10 else None

                if 'Net Profit' in pl_clean.index:
                    profit = pl_clean.loc['Net Profit'].dropna()
                    if len(profit) >= 6 and profit.iloc[-6] > 0:
                        m['Profit CAGR 5Y'] = safe_cagr(
                            profit.iloc[-1], profit.iloc[-6], 5)
                    if len(profit) >= 11 and profit.iloc[-11] > 0:
                        m['Profit CAGR 10Y'] = safe_cagr(
                            profit.iloc[-1], profit.iloc[-11], 10)
                    if len(profit) >= 3:
                        yrs = min(len(profit) - 1, 10)
                        base = profit.iloc[-yrs - 1]
                        if base > 0:
                            m['Profit CAGR Max']  = safe_cagr(
                                profit.iloc[-1], base, yrs)
                            m['Profit CAGR Years'] = yrs

            qr = data.get('Quarterly Results')
            if qr is not None:
                qr_clean = qr.drop(columns=['TTM'], errors='ignore')
                if revenue_row in qr_clean.index:
                    sales_q = qr_clean.loc[revenue_row].dropna()
                    if len(sales_q) >= 4:
                        m['TTM Revenue'] = round(sales_q.iloc[-4:].sum(), 2)
                    if len(sales_q) >= 5:
                        yr_ago = sales_q.iloc[-5]
                        if yr_ago > 0:
                            m['Revenue YoY Q'] = round(
                                (sales_q.iloc[-1] - yr_ago) / yr_ago * 100, 2)
                    yoy = 0
                    for i in range(1, min(5, len(sales_q) - 4)):
                        curr = sales_q.iloc[-i]
                        prev = sales_q.iloc[-i - 4]
                        if prev > 0 and curr > prev:
                            yoy += 1
                        else:
                            break
                    m['Revenue Consecutive YoY'] = yoy
                    if len(sales_q) >= 9:
                        r1 = (sales_q.iloc[-1] - sales_q.iloc[-5]) / sales_q.iloc[-5] * 100
                        r2 = (sales_q.iloc[-5] - sales_q.iloc[-9]) / sales_q.iloc[-9] * 100
                        m['Revenue Accelerating'] = r1 > r2

                if 'Net Profit' in qr_clean.index:
                    profit_q = qr_clean.loc['Net Profit'].dropna()
                    if len(profit_q) >= 4:
                        m['TTM Profit'] = round(profit_q.iloc[-4:].sum(), 2)
                    if len(profit_q) >= 5:
                        yr_ago = profit_q.iloc[-5]
                        if yr_ago > 0:
                            m['Profit YoY Q'] = round(
                                (profit_q.iloc[-1] - yr_ago) / yr_ago * 100, 2)
                    m['Profit Positive Q'] = int((profit_q > 0).sum())
                    m['Total Quarters']    = len(profit_q)

                if margin_row in qr_clean.index:
                    opm_q = qr_clean.loc[margin_row].dropna()
                    m['Latest OPM Q'] = opm_q.iloc[-1] if len(opm_q) > 0 else None
                    m['Avg OPM 4Q']   = round(opm_q.iloc[-4:].mean(), 2) if len(opm_q) >= 4 else None
                    if len(opm_q) >= 5:
                        m['Margin Improving'] = bool(opm_q.iloc[-1] >= opm_q.iloc[-5])

                if 'EPS in Rs' in qr_clean.index:
                    eps_q = qr_clean.loc['EPS in Rs'].dropna()
                    m['Latest EPS Q'] = eps_q.iloc[-1] if len(eps_q) > 0 else None
                    if len(eps_q) >= 5 and eps_q.iloc[-5] > 0:
                        m['EPS YoY Growth'] = round(
                            (eps_q.iloc[-1] - eps_q.iloc[-5]) / eps_q.iloc[-5] * 100, 2)

            bs = data.get('Balance Sheet')
            if bs is not None:
                if debt_row in bs.index:
                    debt = bs.loc[debt_row].dropna()
                    m['Latest Debt']   = debt.iloc[-1] if len(debt) > 0 else None
                    if len(debt) >= 4:
                        m['Debt Reducing'] = bool(debt.iloc[-1] < debt.iloc[-4])
                if 'Reserves' in bs.index and 'Equity Capital' in bs.index:
                    reserves = bs.loc['Reserves'].dropna()
                    eq_cap   = bs.loc['Equity Capital'].dropna()
                    if len(reserves) > 0 and len(eq_cap) > 0:
                        equity           = reserves.iloc[-1] + eq_cap.iloc[-1]
                        m['Latest Equity'] = round(equity, 2)
                        if m.get('TTM Profit') and equity > 0:
                            m['ROE'] = round(m['TTM Profit'] / equity * 100, 2)
                        if m.get('Latest Debt') is not None and equity > 0:
                            m['Debt to Equity'] = round(m['Latest Debt'] / equity, 2)

            cf = data.get('Cash Flows')
            if cf is not None:
                if 'Cash from Operating Activity' in cf.index:
                    op_cf = cf.loc['Cash from Operating Activity'].dropna()
                    m['Latest Operating CF'] = op_cf.iloc[-1] if len(op_cf) > 0 else None
                    m['CF Positive Years']   = int((op_cf > 0).sum())
                    m['CF Total Years']      = len(op_cf)
                    if len(op_cf) >= 4:
                        m['CF Growing'] = bool(op_cf.iloc[-1] > op_cf.iloc[-4])

            ratios = data.get('Ratios')
            if ratios is not None:
                if 'ROCE %' in ratios.index:
                    roce = ratios.loc['ROCE %'].dropna()
                    m['Latest ROCE']  = roce.iloc[-1] if len(roce) > 0 else None
                    if len(roce) >= 4:
                        m['ROCE Improving'] = bool(roce.iloc[-1] > roce.iloc[-4])
                    if len(roce) >= 5:
                        m['Avg ROCE 5Y'] = round(roce.iloc[-5:].mean(), 2)
                if 'ROE %' in ratios.index:
                    roe_s = ratios.loc['ROE %'].dropna()
                    m['ROE from Ratios'] = roe_s.iloc[-1] if len(roe_s) > 0 else None
                    m['ROE Avg 5Y']      = round(roe_s.iloc[-5:].mean(), 2) if len(roe_s) >= 5 else None
                    m['ROE Improving']   = bool(roe_s.iloc[-1] > roe_s.iloc[-4]) if len(roe_s) >= 4 else None

            # Final ROE
            if pd.notna(m.get('ROE from Ratios')):
                m['Final ROE'] = m['ROE from Ratios']
            elif pd.notna(m.get('ROE')):
                m['Final ROE'] = m['ROE']
            else:
                m['Final ROE'] = None

            sh = data.get('Shareholding Pattern')
            if sh is not None:
                if 'Promoters' in sh.index:
                    promoter = sh.loc['Promoters'].dropna()
                    m['Promoter Holding']   = promoter.iloc[-1] if len(promoter) > 0 else None
                    if len(promoter) >= 5:
                        m['Promoter Change 4Q'] = round(promoter.iloc[-1] - promoter.iloc[-5], 2)
                    if len(promoter) >= 9:
                        m['Promoter Change 8Q'] = round(promoter.iloc[-1] - promoter.iloc[-9], 2)
                if 'FIIs' in sh.index:
                    fii = sh.loc['FIIs'].dropna()
                    m['FII Holding']   = fii.iloc[-1] if len(fii) > 0 else None
                    if len(fii) >= 5:
                        m['FII Change 4Q'] = round(fii.iloc[-1] - fii.iloc[-5], 2)
                if 'DIIs' in sh.index:
                    dii = sh.loc['DIIs'].dropna()
                    m['DII Holding']   = dii.iloc[-1] if len(dii) > 0 else None
                    if len(dii) >= 5:
                        m['DII Change 4Q'] = round(dii.iloc[-1] - dii.iloc[-5], 2)

        except Exception as e:
            pass
        return m

    # ── LOAD CHECKPOINT ──────────────────────────────────────
    if os.path.exists(FUND_CHECKPOINT):
        with open(FUND_CHECKPOINT, 'rb') as f:
            ckpt = pickle.load(f)
        all_stock_data = ckpt['raw_data']
        all_metrics    = ckpt['metrics']
        start_idx      = ckpt['next_idx']
        print(f"Resuming from stock {start_idx}/{len(passed_df)}")
    else:
        all_stock_data = {}
        all_metrics    = []
        start_idx      = 0
        print(f"Starting fresh — {len(passed_df)} stocks to scrape")

    failed_stocks = []
    total         = len(passed_df)

    print(f"Already scraped: {len(all_metrics)} stocks")
    print(f"Estimated time : {(total - start_idx) * 3 / 60:.0f} minutes")
    print("-" * 60)

    for i in range(start_idx, total):
        row      = passed_df.iloc[i]
        symbol   = row['Symbol']
        exchange = row['Exchange']

        try:
            # Smart scraper — NSE first, BSE code fallback
            data = scrape_stock(symbol, exchange)
            if data:
                all_stock_data[symbol] = data
                metrics = extract_all_metrics(symbol, data)
                all_metrics.append(metrics)
                sections = len([k for k in data.keys() if k != 'Sector Info'])
                status   = f"✓ ({sections} sections)"
            else:
                failed_stocks.append(symbol)
                status = "✗ No data"

        except Exception as e:
            failed_stocks.append(symbol)
            status = "✗ Error"

        if (i + 1) % 25 == 0 or (i + 1) == total:
            pct = (i + 1) / total * 100
            print(f"  [{i+1:4d}/{total}] {pct:5.1f}% | "
                  f"Scraped: {len(all_metrics)} | Failed: {len(failed_stocks)}")

        if (i + 1) % 50 == 0:
            with open(FUND_CHECKPOINT, 'wb') as f:
                pickle.dump({
                    'raw_data' : all_stock_data,
                    'metrics'  : all_metrics,
                    'next_idx' : i + 1,
                }, f)

        time.sleep(2.5)

    # ── SAVE RESULTS ─────────────────────────────────────────
    with open(FUND_RAW_FILE, 'wb') as f:
        pickle.dump(all_stock_data, f)

    metrics_df = pd.DataFrame(all_metrics)
    metrics_df.to_csv(FUND_OUTPUT, index=False)

    if os.path.exists(FUND_CHECKPOINT):
        os.remove(FUND_CHECKPOINT)

    print(f"\n✅ Fundamental scrape complete")
    print(f"   Scraped  : {len(metrics_df)} stocks")
    print(f"   Failed   : {len(failed_stocks)}")
    print(f"   Columns  : {len(metrics_df.columns)}")
    if failed_stocks:
        print(f"\n   Failed stocks: {failed_stocks[:20]}")
    print(f"\n✅ Saved: {FUND_OUTPUT}")

⚠️  Fundamental scrape already completed — 2142 stocks
    Set FORCE_RESCRAPE = True to re-run quarterly refresh


In [37]:
# ── CELL 10 (FINAL): TIERED QUALITY FILTER ───────────────────

fund_df    = pd.read_csv('../data/fundamental_metrics_full.csv')
prefilt_df = pd.read_csv('../data/prefilt_passed.csv')

fund_df = fund_df.merge(
    prefilt_df[['Symbol', 'Market_Cap_Cr']],
    on='Symbol', how='left'
)

print(f"Total stocks for quality filter: {len(fund_df)}")

def passes_tiered_filter(row):
    mcap   = row.get('Market_Cap_Cr', 0) or 0
    sector = str(row.get('Sector', ''))

    # ── Cap tier thresholds ───────────────────────────────────
    if mcap >= 20000:
        tier         = 'Large Cap'
        min_roe      = 6
        max_de       = 2.5
        min_promo    = 15
        min_rev      = 200
        min_cf_years = 2
    elif mcap >= 5000:
        tier         = 'Mini Large'
        min_roe      = 7
        max_de       = 2.0
        min_promo    = 20
        min_rev      = 100
        min_cf_years = 2
    elif mcap >= 1000:
        tier         = 'Mid Cap'
        min_roe      = 8
        max_de       = 1.5
        min_promo    = 25
        min_rev      = 50
        min_cf_years = 2
    else:
        tier         = 'Small Cap'
        min_roe      = 10
        max_de       = 1.0
        min_promo    = 30
        min_rev      = 20
        min_cf_years = 2

    # ── Company type detection ────────────────────────────────
    financial_sectors = ['Financial Services', 'Banking', 'Insurance']
    is_financial      = any(s in sector for s in financial_sectors)

    lumpy_cf_sectors  = ['Capital Goods', 'Consumer Durables', 'Consumer Services']
    is_lumpy_cf       = any(s in sector for s in lumpy_cf_sectors)

    promoter = float(row.get('Promoter Holding') or 0)
    fii      = float(row.get('FII Holding')      or 0)
    dii      = float(row.get('DII Holding')      or 0)

    is_mnc           = (promoter < 5 and (fii > 40 or dii > 40))
    is_institutional = (promoter < 20 and (fii + dii) > 65)

    # ── Rule 1: TTM Profit > 0 ────────────────────────────────
    profit = row.get('TTM Profit')
    if profit is None or pd.isna(profit) or profit <= 0:
        return False, 'TTM Profit negative or zero', tier

    # ── Rule 2: ROE ───────────────────────────────────────────
    roe = row.get('Final ROE') or row.get('ROE from Ratios') or row.get('ROE')
    if roe is None or pd.isna(roe) or roe < min_roe:
        return False, f'ROE {round(roe,1) if roe else None}% < {min_roe}% ({tier})', tier

    # ── Rule 3: D/E (exempt financials) ──────────────────────
    if not is_financial:
        de = row.get('Debt to Equity')
        if de is not None and not pd.isna(de) and de > max_de:
            return False, f'D/E {de} > {max_de} ({tier})', tier

    # ── Rule 4: Promoter (exempt MNC + institutional) ─────────
    if not is_mnc and not is_institutional:
        promo = row.get('Promoter Holding')
        if promo is None or pd.isna(promo) or promo < min_promo:
            return False, f'Promoter {promo}% < {min_promo}% ({tier})', tier

    # ── Rule 5: Revenue ───────────────────────────────────────
    rev = row.get('TTM Revenue')
    if rev is None or pd.isna(rev) or rev < min_rev:
        return False, f'Revenue {round(rev,0) if rev else None}Cr < {min_rev}Cr ({tier})', tier

    # ── Rule 6: CF (exempt financials) ───────────────────────
    if not is_financial:
        if 'Consumer Durables' in sector:
            pq = row.get('Profit Positive Q') or 0
            tq = row.get('Total Quarters') or 1
            if pd.isna(pq): pq = 0
            if (pq / tq) < 0.8:
                return False, f'Inconsistent profits: {pq}/{tq} quarters', tier
        else:
            cf_pos   = row.get('CF Positive Years') or 0
            cf_total = row.get('CF Total Years') or 1
            if pd.isna(cf_pos): cf_pos = 0
            if (cf_pos / cf_total) < 0.60:
                return False, f'Poor CF: {cf_pos}/{cf_total} yrs', tier

    # ── Rule 7: Promoter not selling > 5% in 4Q ──────────────
    if not is_mnc and not is_institutional:
        p_change = row.get('Promoter Change 4Q')
        if p_change is not None and not pd.isna(p_change):
            if p_change < -5:
                return False, f'Promoter selling: {p_change}% in 4Q', tier

    # ── Rule 8: Profit positive ≥ 80% quarters ───────────────
    pq = row.get('Profit Positive Q') or 0
    tq = row.get('Total Quarters') or 1
    if pd.isna(pq): pq = 0
    if tq > 0 and (pq / tq) < 0.8:
        return False, f'Inconsistent profits: {pq}/{tq} quarters', tier

    return True, None, tier

# ── APPLY FILTER ─────────────────────────────────────────────
results = []
for _, row in fund_df.iterrows():
    passed, reason, tier = passes_tiered_filter(row)
    results.append({
        'Symbol'       : row['Symbol'],
        'Sector'       : row.get('Sector'),
        'Market_Cap_Cr': row.get('Market_Cap_Cr'),
        'Cap_Tier'     : tier,
        'Passed'       : passed,
        'Fail_Reason'  : reason,
    })

results_df     = pd.DataFrame(results)
passed_quality = results_df[results_df['Passed'] == True]
failed_quality = results_df[results_df['Passed'] == False]

# Save
passed_quality[['Symbol']].to_csv('../data/quality_passed.csv', index=False)
results_df.to_csv('../data/quality_filter_results.csv', index=False)

print(f"\n✅ Tiered quality filter complete")
print(f"   Total input : {len(results_df)}")
print(f"   Passed      : {len(passed_quality)}")
print(f"   Failed      : {len(failed_quality)}")

print(f"\nPassed by tier:")
print(passed_quality['Cap_Tier'].value_counts().to_string())

print(f"\nTop fail reasons:")
print(failed_quality['Fail_Reason']
      .str.split('<').str[0].str.strip()
      .value_counts().head(10).to_string())

print(f"\nPassed by sector (top 15):")
print(passed_quality['Sector'].value_counts().head(15).to_string())



Total stocks for quality filter: 2142

✅ Tiered quality filter complete
   Total input : 2142
   Passed      : 752
   Failed      : 1390

Passed by tier:
Cap_Tier
Large Cap     214
Mid Cap       210
Mini Large    201
Small Cap     127

Top fail reasons:
Fail_Reason
TTM Profit negative or zero    561
ROE nan%                        29
Promoter nan%                   20
ROE 5.6%                        14
Poor CF: 7.0/12.0 yrs           13
ROE 5.8%                        12
ROE 3.0%                        12
ROE 6.8%                        11
ROE 5.7%                        10
ROE 6.6%                        10

Passed by sector (top 15):
Sector
Capital Goods                     130
Financial Services                 91
Healthcare                         68
Information Technology             60
Automobile and Auto Components     59
Chemicals                          57
Fast Moving Consumer Goods         54
Consumer Durables                  40
Consumer Services                  26
Constru

In [58]:
# ── CELL 11: PRICE HISTORY FETCH (FULL UNIVERSE) ─────────────
# ── RE-DOWNLOAD ALL PRICES SEQUENTIALLY ──────────────────────
# Parallel processing caused data mixing — must be sequential

PRICE_FILE = '../data/price_data_full.pkl'

print(f"Re-downloading price data for all stocks sequentially...")
print(f"Estimated time: ~45-60 minutes")
print("-" * 60)

quality_df = pd.read_csv('../data/quality_passed.csv')
prefilt_df = pd.read_csv('../data/prefilt_passed.csv')

ticker_map = {}
for _, row in prefilt_df.iterrows():
    sym = row['Symbol']
    ticker_map[sym] = f"{sym}.NS" if row['Exchange'] == 'NSE' else f"{sym}.BO"

symbols    = quality_df['Symbol'].tolist()
price_data = {}
failed     = []

for i, symbol in enumerate(symbols):
    ticker = ticker_map.get(symbol, f"{symbol}.NS")
    try:
        df = yf.download(ticker, period='max', interval='1d',
                         progress=False, auto_adjust=True)
        # Flatten MultiIndex
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] for col in df.columns]
        # Remove duplicate columns
        df = df.loc[:, ~df.columns.duplicated()]

        if df is not None and len(df) >= 50:
            price_data[symbol] = df
        else:
            failed.append(symbol)

    except Exception as e:
        failed.append(symbol)

    # Progress every 50
    if (i + 1) % 50 == 0 or (i + 1) == len(symbols):
        pct = (i + 1) / len(symbols) * 100
        print(f"  [{i+1:4d}/{len(symbols)}] {pct:5.1f}% | "
              f"Fetched: {len(price_data)} | Failed: {len(failed)}")

    time.sleep(0.5)  # sequential — safe rate

# Save
with open(PRICE_FILE, 'wb') as f:
    pickle.dump(price_data, f)

print(f"\n✅ Price data saved")
print(f"   Fetched : {len(price_data)} stocks")
print(f"   Failed  : {len(failed)}")
if failed:
    print(f"   Failed  : {failed[:20]}")

Re-downloading price data for all stocks sequentially...
Estimated time: ~45-60 minutes
------------------------------------------------------------
  [  50/752]   6.6% | Fetched: 50 | Failed: 0
  [ 100/752]  13.3% | Fetched: 100 | Failed: 0
  [ 150/752]  19.9% | Fetched: 150 | Failed: 0
  [ 200/752]  26.6% | Fetched: 200 | Failed: 0
  [ 250/752]  33.2% | Fetched: 250 | Failed: 0
  [ 300/752]  39.9% | Fetched: 300 | Failed: 0
  [ 350/752]  46.5% | Fetched: 350 | Failed: 0
  [ 400/752]  53.2% | Fetched: 400 | Failed: 0
  [ 450/752]  59.8% | Fetched: 450 | Failed: 0
  [ 500/752]  66.5% | Fetched: 500 | Failed: 0
  [ 550/752]  73.1% | Fetched: 550 | Failed: 0
  [ 600/752]  79.8% | Fetched: 600 | Failed: 0
  [ 650/752]  86.4% | Fetched: 650 | Failed: 0
  [ 700/752]  93.1% | Fetched: 700 | Failed: 0
  [ 750/752]  99.7% | Fetched: 750 | Failed: 0
  [ 752/752] 100.0% | Fetched: 752 | Failed: 0

✅ Price data saved
   Fetched : 752 stocks
   Failed  : 0


In [59]:
# ── CELL 12b: FUNDAMENTAL SCORING — FULL UNIVERSE ────────────

fund_full  = pd.read_csv('../data/fundamental_metrics_full.csv')
quality_df = pd.read_csv('../data/quality_passed.csv')

# Filter to quality-passed stocks only
fund_df = fund_full[fund_full['Symbol'].isin(quality_df['Symbol'])].copy().reset_index(drop=True)
print(f"Stocks to score: {len(fund_df)}")

# ── SCORING HELPER FUNCTIONS (from Day 4) ────────────────────
def score_metric_level(value, thresholds, reverse=False):
    if pd.isna(value):
        return None
    if reverse:
        if value <= thresholds[0]:   return 3
        elif value <= thresholds[1]: return 2
        elif value <= thresholds[2]: return 1
        else:                        return 0
    else:
        if value >= thresholds[0]:   return 3
        elif value >= thresholds[1]: return 2
        elif value >= thresholds[2]: return 1
        else:                        return 0

def calculate_historical_score(row):
    score   = 0
    cagr_years = float(row.get('Revenue CAGR Years') or row.get('Profit CAGR Years') or 0)
    if cagr_years < 5:   discount = 0.50
    elif cagr_years < 8: discount = 0.20
    else:                discount = 0.00

    rev_growth = float(row.get('Revenue CAGR 5Y') or row.get('Revenue CAGR 10Y') or row.get('Revenue CAGR Max') or 0)
    if rev_growth >= 20:   s = 15
    elif rev_growth >= 15: s = 12
    elif rev_growth >= 10: s = 9
    elif rev_growth >= 5:  s = 5
    else:                  s = 0
    score += s

    profit_growth = float(row.get('Profit CAGR 5Y') or row.get('Profit CAGR 10Y') or row.get('Profit CAGR Max') or 0)
    if profit_growth >= 20:  s = 15
    elif profit_growth >= 15:s = 12
    elif profit_growth >= 10:s = 9
    elif profit_growth >= 5: s = 5
    else:                    s = 0
    score += s

    opm = float(row.get('Avg OPM 5Y') or row.get('Avg OPM 10Y') or 0)
    if opm >= 25:   s = 5
    elif opm >= 20: s = 4
    elif opm >= 15: s = 3
    elif opm >= 10: s = 2
    elif opm >= 5:  s = 1
    else:           s = 0
    score += s

    roe = float(row.get('Final ROE') or row.get('ROE') or 0)
    if roe >= 25:   s = 5
    elif roe >= 20: s = 4
    elif roe >= 15: s = 3
    elif roe >= 10: s = 2
    else:           s = 0
    score += s

    return min(round(score * (1 - discount), 1), 40)

def calculate_financial_score(row):
    score      = 0
    cagr_years = float(row.get('Revenue CAGR Years') or row.get('Profit CAGR Years') or 0)
    if cagr_years < 5:   discount = 0.50
    elif cagr_years < 8: discount = 0.20
    else:                discount = 0.00

    roe = float(row.get('Final ROE') or row.get('ROE') or 0)
    if roe >= 20:   s = 20
    elif roe >= 18: s = 17
    elif roe >= 15: s = 13
    elif roe >= 12: s = 8
    elif roe >= 10: s = 4
    else:           s = 0
    score += s

    profit_growth = float(row.get('Profit CAGR 5Y') or row.get('Profit CAGR 10Y') or row.get('Profit CAGR Max') or 0)
    if profit_growth >= 20:  s = 10
    elif profit_growth >= 15:s = 8
    elif profit_growth >= 10:s = 6
    elif profit_growth >= 5: s = 3
    else:                    s = 0
    score += s

    rev_growth = float(row.get('Revenue CAGR 5Y') or row.get('Revenue CAGR 10Y') or row.get('Revenue CAGR Max') or 0)
    if rev_growth >= 20:  s = 10
    elif rev_growth >= 15:s = 8
    elif rev_growth >= 10:s = 6
    elif rev_growth >= 5: s = 3
    else:                 s = 0
    score += s

    return min(round(score * (1 - discount), 1), 40)

def calculate_peer_scores(df):
    peer_scores  = []
    peer_metrics = {
        'Final ROE'        : True,
        'Revenue CAGR Max' : True,
        'Profit CAGR Max'  : True,
        'Avg OPM 5Y'       : True,
        'Latest ROCE'      : True,
        'Debt to Equity'   : False,
        'Revenue YoY Q'    : True,
        'Profit YoY Q'     : True,
    }
    for _, row in df.iterrows():
        symbol       = row['Symbol']
        sector       = row.get('Sector', 'Unknown')
        scores       = {}
        sector_peers = df[df['Sector'] == sector]
        for metric, higher_is_better in peer_metrics.items():
            stock_val = row.get(metric)
            if pd.isna(stock_val):
                scores[metric] = None
                continue
            peer_vals = sector_peers[metric].dropna()
            if len(peer_vals) < 2:
                scores[metric] = 2
                continue
            if higher_is_better:
                percentile = (peer_vals < stock_val).sum() / len(peer_vals) * 100
            else:
                percentile = (peer_vals > stock_val).sum() / len(peer_vals) * 100
            if percentile >= 80:   score = 5
            elif percentile >= 60: score = 4
            elif percentile >= 40: score = 3
            elif percentile >= 20: score = 2
            else:                  score = 1
            scores[metric] = score
        valid = [s for s in scores.values() if s is not None]
        if valid:
            total = round(sum(valid) / (len(valid) * 5) * 40, 1)
        else:
            total = 20
        peer_scores.append({'Symbol': symbol, 'Sector': sector, 'Peer Score': total})
    return pd.DataFrame(peer_scores)

def calculate_quality_score(row):
    scores = {}
    promoter = row.get('Promoter Holding')
    level    = score_metric_level(promoter, [60, 50, 40])
    scores['Promoter Holding'] = level if level is not None else 0

    promoter_change = float(row.get('Promoter Change 4Q') or 0)
    if pd.isna(promoter_change): promoter_change = 0
    if promoter_change > 1:      pt_score = 4
    elif promoter_change > 0:    pt_score = 3
    elif promoter_change >= -1:  pt_score = 2
    elif promoter_change >= -3:  pt_score = 1
    else:                        pt_score = 0
    scores['Promoter Trend'] = pt_score

    fii = float(row.get('FII Holding') or 0)
    dii = float(row.get('DII Holding') or 0)
    if pd.isna(fii): fii = 0
    if pd.isna(dii): dii = 0
    institutional = fii + dii
    if institutional < 10:   inst_score = 4
    elif institutional < 20: inst_score = 3
    elif institutional < 35: inst_score = 2
    else:                    inst_score = 1
    scores['Institutional'] = inst_score

    profit_q    = row.get('Profit Positive Q', 0) or 0
    total_q     = row.get('Total Quarters',    1) or 1
    consistency = (profit_q / total_q * 100) if total_q > 0 else 0
    if consistency >= 95:   pc_score = 4
    elif consistency >= 85: pc_score = 3
    elif consistency >= 75: pc_score = 2
    else:                   pc_score = 1
    scores['Profit Consistency'] = pc_score

    cf_years   = row.get('CF Positive Years', 0) or 0
    cf_total   = row.get('CF Total Years',    1) or 1
    cf_ratio   = (cf_years / cf_total * 100) if cf_total > 0 else 0
    cf_growing = row.get('CF Growing')
    if cf_ratio >= 90 and cf_growing == True: cq_score = 4
    elif cf_ratio >= 80:                      cq_score = 3
    elif cf_ratio >= 65:                      cq_score = 2
    else:                                     cq_score = 1
    scores['CF Quality'] = cq_score

    return {
        'Symbol'         : row['Symbol'],
        'Sector'         : row.get('Sector'),
        'Quality Score'  : sum(scores.values()),
        'Promoter Holding %': promoter,
        'FII + DII %'    : round(institutional, 2),
    }

# ── RUN ALL THREE SCORES ──────────────────────────────────────
financial_sectors = ['Financial Services', 'Banking', 'Insurance']

historical_scores = []
for _, row in fund_df.iterrows():
    sector = str(row.get('Sector', ''))
    is_fin = any(kw in sector for kw in financial_sectors)
    if is_fin:
        hist_score = calculate_financial_score(row)
    else:
        hist_score = calculate_historical_score(row)
    historical_scores.append({
        'Symbol'          : row['Symbol'],
        'Sector'          : sector,
        'Historical Score': hist_score,
    })

historical_df = pd.DataFrame(historical_scores)
peer_df       = calculate_peer_scores(fund_df)
quality_scores = []
for _, row in fund_df.iterrows():
    quality_scores.append(calculate_quality_score(row))
quality_df_scores = pd.DataFrame(quality_scores)

# ── COMBINE → FINAL SCORE ─────────────────────────────────────
final_df = historical_df[['Symbol', 'Sector', 'Historical Score']].merge(
    peer_df[['Symbol', 'Peer Score']], on='Symbol'
).merge(
    quality_df_scores[['Symbol', 'Quality Score',
                        'Promoter Holding %', 'FII + DII %']], on='Symbol'
)

final_df['Final Score'] = (
    final_df['Historical Score'] +
    final_df['Peer Score']       +
    final_df['Quality Score']
).round(1)

final_df = final_df.sort_values('Final Score', ascending=False).reset_index(drop=True)

# Save
final_df.to_csv('../data/fundamental_scores_full.csv', index=False)

print(f"✅ Fundamental scoring complete")
print(f"   Stocks scored : {len(final_df)}")
print(f"\nTop 20:")
for i, row in final_df.head(20).iterrows():
    print(f"  {i+1:3}. {row['Symbol']:<15} {row['Final Score']:>6}/100  {row['Sector']}")

Stocks to score: 752
✅ Fundamental scoring complete
   Stocks scored : 752

Top 20:
    1. SOLARINDS         89.0/100  Chemicals
    2. BLS               89.0/100  Consumer Services
    3. ALKYLAMINE        87.0/100  Chemicals
    4. KPEL              87.0/100  Power
    5. 3BBLACKBIO        86.7/100  Healthcare
    6. SIGMA             86.0/100  Information Technology
    7. NAM-INDIA         86.0/100  Financial Services
    8. INDIAMART         85.6/100  Consumer Services
    9. CDSL              85.0/100  Financial Services
   10. CAPLIPOINT        85.0/100  Healthcare
   11. NATCOPHARM        84.0/100  Healthcare
   12. LLOYDSME          84.0/100  Metals & Mining
   13. SANDUMA           83.7/100  Oil, Gas & Consumable Fuels
   14. SHARDAMOTR        83.0/100  Automobile and Auto Components
   15. FCL               82.9/100  Chemicals
   16. SIKA              82.1/100  Capital Goods
   17. BANCOINDIA        82.0/100  Automobile and Auto Components
   18. KSOLVES           82.0/100  

In [61]:
# ── CELL 12: TECHNICAL ANALYSIS — FULL UNIVERSE ──────────────
# Exact same logic as Day 5, scaled to 752 stocks
# Saves checkpoint every 50 stocks

TECH_CHECKPOINT = '../data/tech_checkpoint.pkl'
TECH_OUTPUT     = '../data/technical_report_full.csv'
INDICATOR_FILE  = '../data/indicator_data_full.pkl'
FORCE_RERUN     = False

if not FORCE_RERUN and os.path.exists(TECH_OUTPUT):
    existing = pd.read_csv(TECH_OUTPUT)
    print(f"⚠️  Technical analysis already done — {len(existing)} stocks")
    print(f"    Set FORCE_RERUN = True to re-run")

else:
    # ── LOAD DATA ─────────────────────────────────────────────
    with open('../data/price_data_full.pkl', 'rb') as f:
        price_data_full = pickle.load(f)

    quality_df  = pd.read_csv('../data/quality_passed.csv')
    fund_full   = pd.read_csv('../data/fundamental_scores_full.csv')
    prefilt_df  = pd.read_csv('../data/prefilt_passed.csv')

    # Merge market cap
    fund_full = fund_full.merge(
        prefilt_df[['Symbol', 'Market_Cap_Cr']], on='Symbol', how='left'
    )

    symbols = quality_df['Symbol'].tolist()
    print(f"Stocks to analyse : {len(symbols)}")

    # ── FIX DUPLICATE COLUMNS IN PRICE DATA ──────────────────
    fixed = 0
    for sym in price_data_full:
        df = price_data_full[sym]
        if df.columns.duplicated().any():
            price_data_full[sym] = df.loc[:, ~df.columns.duplicated()]
            fixed += 1
    if fixed > 0:
        print(f"✅ Fixed {fixed} stocks with duplicate columns")
        with open('../data/price_data_full.pkl', 'wb') as f:
            pickle.dump(price_data_full, f)

    # ── INDICATOR FUNCTIONS ───────────────────────────────────
    def calculate_indicators(df):
        data = df.copy()
        data['EMA20']  = data['Close'].ewm(span=20,  adjust=False).mean()
        data['EMA50']  = data['Close'].ewm(span=50,  adjust=False).mean()
        data['EMA200'] = data['Close'].ewm(span=200, adjust=False).mean()

        delta    = data['Close'].diff()
        gain     = delta.where(delta > 0, 0)
        loss     = -delta.where(delta < 0, 0)
        avg_gain = gain.ewm(span=14, adjust=False).mean()
        avg_loss = loss.ewm(span=14, adjust=False).mean()
        rs       = avg_gain / avg_loss
        data['RSI'] = 100 - (100 / (1 + rs))

        ema12             = data['Close'].ewm(span=12, adjust=False).mean()
        ema26             = data['Close'].ewm(span=26, adjust=False).mean()
        data['MACD']      = ema12 - ema26
        data['Signal']    = data['MACD'].ewm(span=9, adjust=False).mean()
        data['MACD_Hist'] = data['MACD'] - data['Signal']

        high     = data['High']
        low      = data['Low']
        close    = data['Close']
        tr1      = high - low
        tr2      = (high - close.shift()).abs()
        tr3      = (low  - close.shift()).abs()
        tr       = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        dm_plus  = high.diff()
        dm_minus = -low.diff()
        dm_plus  = dm_plus.where((dm_plus > dm_minus) & (dm_plus > 0), 0)
        dm_minus = dm_minus.where((dm_minus > dm_plus) & (dm_minus > 0), 0)
        atr      = tr.ewm(span=14, adjust=False).mean()
        di_plus  = 100 * dm_plus.ewm(span=14, adjust=False).mean() / atr
        di_minus = 100 * dm_minus.ewm(span=14, adjust=False).mean() / atr
        dx       = 100 * (di_plus - di_minus).abs() / (di_plus + di_minus)
        data['ADX']      = dx.ewm(span=14, adjust=False).mean()
        data['DI_Plus']  = di_plus
        data['DI_Minus'] = di_minus

        data['Vol_MA20']  = data['Volume'].rolling(20).mean()
        data['Vol_Ratio'] = data['Volume'] / data['Vol_MA20']
        return data

    def calculate_volume_profile(df, lookback_days=252, bins=30):
        recent    = df.tail(lookback_days).copy()
        price_min = recent['Low'].min()
        price_max = recent['High'].max()
        if price_max <= price_min:
            return None
        price_bins      = np.linspace(price_min, price_max, bins + 1)
        volume_at_price = np.zeros(bins)
        for _, row in recent.iterrows():
            candle_low   = row['Low']
            candle_high  = row['High']
            candle_vol   = row['Volume']
            candle_range = candle_high - candle_low
            if candle_range == 0:
                continue
            for i in range(bins):
                bin_low      = price_bins[i]
                bin_high     = price_bins[i + 1]
                overlap_low  = max(candle_low,  bin_low)
                overlap_high = min(candle_high, bin_high)
                if overlap_high > overlap_low:
                    overlap_pct          = (overlap_high - overlap_low) / candle_range
                    volume_at_price[i]  += candle_vol * overlap_pct
        poc_idx   = np.argmax(volume_at_price)
        poc_price = (price_bins[poc_idx] + price_bins[poc_idx + 1]) / 2
        total_vol  = volume_at_price.sum()
        target_vol = total_vol * 0.70
        lower_idx  = poc_idx
        upper_idx  = poc_idx
        va_vol     = volume_at_price[poc_idx]
        while va_vol < target_vol:
            can_up   = upper_idx + 1 < bins
            can_down = lower_idx - 1 >= 0
            exp_up   = volume_at_price[upper_idx + 1] if can_up   else 0
            exp_down = volume_at_price[lower_idx - 1] if can_down else 0
            if not can_up and not can_down:
                break
            if exp_up >= exp_down and can_up:
                upper_idx += 1
                va_vol    += exp_up
            elif can_down:
                lower_idx -= 1
                va_vol    += exp_down
            else:
                break
        val           = (price_bins[lower_idx] + price_bins[lower_idx + 1]) / 2
        vah           = (price_bins[upper_idx] + price_bins[upper_idx + 1]) / 2
        max_vol       = volume_at_price.max()
        edge_vol      = (volume_at_price[:3].mean() + volume_at_price[-3:].mean()) / 2
        is_bell       = max_vol > edge_vol * 2.5
        current_price = df['Close'].iloc[-1]
        if current_price > vah * 1.02:      breakout_status = 'BREAKOUT_UP'
        elif current_price < val * 0.98:    breakout_status = 'BREAKOUT_DOWN'
        elif current_price > vah * 0.98:    breakout_status = 'AT_RESISTANCE'
        else:                               breakout_status = 'INSIDE'
        return {
            'poc_price': round(poc_price, 2), 'val': round(val, 2),
            'vah': round(vah, 2), 'is_bell': is_bell,
            'breakout_status': breakout_status,
            'current_price': round(current_price, 2),
        }

    def score_momentum(df, vp):
        scores = {}
        latest = df.iloc[-1]
        close  = latest['Close']
        ema20  = latest['EMA20']
        ema50  = latest['EMA50']
        ema200 = latest['EMA200']
        if close > ema20 > ema50 > ema200:  scores['EMA'] = 30
        elif close > ema50 > ema200:        scores['EMA'] = 20
        elif close > ema200:                scores['EMA'] = 10
        else:                               scores['EMA'] = 0
        rsi = latest['RSI']
        if 50 <= rsi <= 70:     scores['RSI'] = 20
        elif 45 <= rsi < 50:    scores['RSI'] = 12
        elif 70 < rsi <= 80:    scores['RSI'] = 8
        else:                   scores['RSI'] = 0
        macd   = latest['MACD']
        signal = latest['Signal']
        hist   = latest['MACD_Hist']
        if macd > signal and hist > 0 and macd > 0:  scores['MACD'] = 20
        elif macd > signal and hist > 0:             scores['MACD'] = 15
        elif macd > signal:                          scores['MACD'] = 10
        else:                                        scores['MACD'] = 0
        adx      = latest['ADX']
        di_plus  = latest['DI_Plus']
        di_minus = latest['DI_Minus']
        if adx > 25 and di_plus > di_minus:   scores['ADX'] = 15
        elif adx > 20 and di_plus > di_minus: scores['ADX'] = 10
        elif adx > 25:                        scores['ADX'] = 5
        else:                                 scores['ADX'] = 0
        if vp:
            status = vp['breakout_status']
            if status == 'BREAKOUT_UP':     scores['VP'] = 15
            elif status == 'AT_RESISTANCE': scores['VP'] = 10
            elif status == 'INSIDE':        scores['VP'] = 5
            else:                           scores['VP'] = 0
        else:
            scores['VP'] = 5
        return sum(scores.values()), scores

    def score_reversal(df, vp, lookback=20):
        scores = {}
        if len(df) < lookback + 10:
            return 0, {}
        latest_rsi = df['RSI'].iloc[-1]
        if latest_rsi < 35:    scores['RSI_Level'] = 20
        elif latest_rsi < 45:  scores['RSI_Level'] = 10
        else:                  scores['RSI_Level'] = 0
        rsi_5d    = df['RSI'].tail(5).min()
        rsi_20d   = df['RSI'].tail(20).min()
        price_5d  = df['Close'].tail(5).min()
        price_20d = df['Close'].tail(20).min()
        scores['RSI_Divergence'] = 15 if (price_5d <= price_20d and rsi_5d > rsi_20d) else 0
        latest_hist = df['MACD_Hist'].iloc[-1]
        prev_hist   = df['MACD_Hist'].iloc[-5]
        if latest_hist > 0 and prev_hist < 0:
            scores['MACD_Cross'] = 20; scores['MACD_Rising'] = 0
        elif latest_hist > prev_hist and latest_hist < 0:
            scores['MACD_Cross'] = 0;  scores['MACD_Rising'] = 15
        else:
            scores['MACD_Cross'] = 0;  scores['MACD_Rising'] = 0
        macd_5d  = df['MACD_Hist'].tail(5).min()
        macd_20d = df['MACD_Hist'].tail(20).min()
        scores['MACD_Divergence'] = 15 if (price_5d <= price_20d and macd_5d > macd_20d) else 0
        adx_latest = df['ADX'].iloc[-1]
        adx_10d    = df['ADX'].iloc[-10]
        if adx_latest < adx_10d and adx_latest > 20:  scores['ADX'] = 15
        elif adx_latest < adx_10d:                    scores['ADX'] = 8
        else:                                          scores['ADX'] = 0
        vol_ratio = df['Vol_Ratio'].iloc[-1]
        if vol_ratio < 0.6:    scores['Volume'] = 15
        elif vol_ratio < 0.8:  scores['Volume'] = 8
        else:                  scores['Volume'] = 0
        return min(sum(scores.values()), 100), scores

    def detect_consolidation_dynamic(df, min_days=300, max_range_pct=50,
                                      max_total_drift=10, inside_pct_required=0.82):
        close = df['Close']
        n     = len(df)
        if n < min_days + 5:
            return {'found': False, 'is_valid': False, 'consol_days': 0}
        best_result = None
        for lookback in range(min_days, min(800, n - 5)):
            start_idx  = n - lookback
            segment    = df.iloc[start_idx:n]
            range_high = segment['High'].quantile(0.95)
            range_low  = segment['Low'].quantile(0.05)
            range_pct  = (range_high - range_low) / range_low * 100
            rng        = range_high - range_low
            if range_pct > max_range_pct or rng == 0:
                continue
            buffer      = 0.03
            bars_inside = (
                (segment['High'] <= range_high * (1 + buffer)) &
                (segment['Low']  >= range_low  * (1 - buffer))
            ).sum()
            pct_inside = bars_inside / len(segment)
            if pct_inside < inside_pct_required:
                continue
            x        = np.arange(len(segment))
            hi_drift = abs(np.polyfit(x, segment['High'].values,  1)[0] / segment['High'].mean()  * 100) * lookback
            lo_drift = abs(np.polyfit(x, segment['Low'].values,   1)[0] / segment['Low'].mean()   * 100) * lookback
            cl_drift = abs(np.polyfit(x, segment['Close'].values, 1)[0] / segment['Close'].mean() * 100) * lookback
            if hi_drift > max_total_drift or lo_drift > max_total_drift or cl_drift > max_total_drift:
                continue
            price_start  = segment['Close'].iloc[0]
            price_end    = segment['Close'].iloc[-1]
            price_change = (price_end - price_start) / price_start * 100
            if abs(price_change) > 25:
                continue
            end_vs_range = (price_end - range_low) / rng * 100
            if end_vs_range < -10:
                continue
            close_arr    = segment['Close'].values
            peak_idx     = np.argmax(close_arr)
            peak_pos     = peak_idx / lookback * 100
            peak_price   = close_arr.max()
            peak_vs_rng  = (peak_price  - range_low) / rng * 100
            start_vs_rng = (price_start - range_low) / rng * 100
            if peak_pos > 20 and peak_pos < 80 and start_vs_rng < 35 and end_vs_range < 35 and peak_vs_rng > 90:
                continue
            pre_start   = max(0, start_idx - 120)
            pre_segment = df.iloc[pre_start:start_idx]
            pre_close   = pre_segment['Close'].values
            if len(pre_close) > 5:
                pre_x    = np.arange(len(pre_close))
                pre_norm = np.polyfit(pre_x, pre_close, 1)[0] / pre_close.mean() * 100
                if pre_norm < (-0.15 if lookback < 365 else -0.60):
                    continue
                pre_high   = pre_segment['High'].max()
                drop_pct   = (range_high - pre_high) / pre_high * 100
                if drop_pct < (-20 if lookback < 365 else -60):
                    continue
            if lookback > (best_result['consol_days'] if best_result else 0):
                best_result = {
                    'consol_days': lookback,
                    'range_high' : round(range_high, 2),
                    'range_low'  : round(range_low,  2),
                    'range_pct'  : round(range_pct,  2),
                    'pct_inside' : round(pct_inside,  3),
                    'start_idx'  : start_idx,
                }
        if best_result is None:
            return {'found': False, 'is_valid': False, 'consol_days': 0}
        consol_days   = best_result['consol_days']
        range_high    = best_result['range_high']
        range_low     = best_result['range_low']
        current_price = close.iloc[-1]
        consol_df     = df.iloc[best_result['start_idx']:n]
        res_touches   = int((consol_df['High'] >= range_high * 0.98).sum())
        sup_touches   = int((consol_df['Low']  <= range_low  * 1.02).sum())
        pct_above     = (current_price - range_high) / range_high * 100
        is_breaking   = current_price >= range_high * 0.98
        vol_ratio     = round(df['Vol_Ratio'].tail(5).max(), 2)
        vp            = calculate_volume_profile(consol_df, lookback_days=consol_days, bins=20)
        has_bell      = vp['is_bell'] if vp else False
        if consol_days < 365:   duration_label = f'{consol_days}d (under 1 year)'
        elif consol_days < 500: duration_label = f'{consol_days}d (1-1.5 years)'
        elif consol_days < 730: duration_label = f'{consol_days}d (1.5-2 years)'
        else:                   duration_label = f'{consol_days}d (2+ years)'
        is_valid = res_touches >= 2 and sup_touches >= 2
        return {
            'found'           : True,
            'is_valid'        : is_valid,
            'consol_days'     : consol_days,
            'duration_label'  : duration_label,
            'range_high'      : range_high,
            'range_low'       : range_low,
            'range_pct'       : best_result['range_pct'],
            'pct_above'       : round(pct_above, 2),
            'is_breaking_out' : is_breaking,
            'breakout_volume' : vol_ratio,
            'has_bell'        : has_bell,
            'current_price'   : round(current_price, 2),
            'resistance_touches': res_touches,
            'support_touches'   : sup_touches,
        }

    def get_consolidation_info(df):
        result = detect_consolidation_dynamic(df)
        if not result.get('found') or not result.get('is_valid'):
            return {
                'consolidating': False, 'consol_days': 0,
                'duration_label': 'No consolidation',
                'range_high': None, 'range_low': None, 'range_pct': None,
                'pct_to_breakout': None, 'is_breaking_out': False,
                'breakout_volume': None, 'has_bell': False,
                'resistance_touches': 0, 'support_touches': 0,
                'current_price': df['Close'].iloc[-1],
            }
        return {
            'consolidating'     : True,
            'consol_days'       : result['consol_days'],
            'duration_label'    : result['duration_label'],
            'range_high'        : result['range_high'],
            'range_low'         : result['range_low'],
            'range_pct'         : result['range_pct'],
            'pct_to_breakout'   : result['pct_above'],
            'is_breaking_out'   : result['is_breaking_out'],
            'breakout_volume'   : result['breakout_volume'],
            'has_bell'          : result['has_bell'],
            'resistance_touches': result['resistance_touches'],
            'support_touches'   : result['support_touches'],
            'current_price'     : result['current_price'],
        }

    def classify_mcap(mcap_cr):
        if mcap_cr >= 20000:  return 'Large Cap'
        elif mcap_cr >= 5000: return 'Mini Large Cap'
        elif mcap_cr >= 1000: return 'Mid Cap'
        else:                 return 'Small Cap'

    # ── CALCULATE INDICATORS ─────────────────────────────────
    print("Calculating indicators...")
    indicator_data_full = {}
    for symbol in symbols:
        if symbol not in price_data_full:
            continue
        try:
            df = price_data_full[symbol]
            indicator_data_full[symbol] = calculate_indicators(df)
        except Exception as e:
            pass

    print(f"✅ Indicators done: {len(indicator_data_full)} stocks")
    with open(INDICATOR_FILE, 'wb') as f:
        pickle.dump(indicator_data_full, f)

    # ── LOAD CHECKPOINT ──────────────────────────────────────
    if os.path.exists(TECH_CHECKPOINT):
        with open(TECH_CHECKPOINT, 'rb') as f:
            ckpt = pickle.load(f)
        tech_reports = ckpt['reports']
        done_syms    = ckpt['done']
        print(f"Resuming — {len(done_syms)} done")
    else:
        tech_reports = []
        done_syms    = set()

    remaining = [s for s in symbols
                 if s in indicator_data_full and s not in done_syms]
    print(f"Remaining : {len(remaining)} stocks")
    print("-" * 60)

    for i, symbol in enumerate(remaining):
        try:
            df       = indicator_data_full[symbol]
            latest   = df.iloc[-1]
            fund_row = fund_full[fund_full['Symbol'] == symbol]

            fund_score = float(fund_row['Final Score'].values[0]) \
                         if len(fund_row) > 0 and 'Final Score' in fund_row.columns \
                         else 50.0
            sector     = fund_row['Sector'].values[0] \
                         if len(fund_row) > 0 else 'Unknown'
            mcap_cr    = float(fund_row['Market_Cap_Cr'].values[0]) \
                         if len(fund_row) > 0 and 'Market_Cap_Cr' in fund_row.columns \
                         else 0

            close    = latest['Close']
            rsi      = round(latest['RSI'],       2)
            adx      = round(latest['ADX'],       2)
            hist     = round(latest['MACD_Hist'], 2)
            vol_r    = round(latest['Vol_Ratio'], 2)
            ema20    = round(latest['EMA20'],     2)
            ema50    = round(latest['EMA50'],     2)
            ema200   = round(latest['EMA200'],    2)
            di_plus  = round(latest['DI_Plus'],   2)
            di_minus = round(latest['DI_Minus'],  2)

            vp           = calculate_volume_profile(df)
            mom_score, _ = score_momentum(df, vp)
            rev_score, _ = score_reversal(df, vp)
            consol_info  = get_consolidation_info(df)

            if mom_score >= rev_score and mom_score >= 50:
                best_setup = 'Momentum'
                tech_score = mom_score
            elif rev_score >= mom_score and rev_score >= 50:
                best_setup = 'Reversal'
                tech_score = rev_score
            else:
                best_setup = 'Watching'
                tech_score = max(mom_score, rev_score)

            in_consol       = consol_info['consolidating']
            consol_days     = consol_info['consol_days']
            consol_label    = consol_info['duration_label']
            pct_to_breakout = consol_info['pct_to_breakout'] or 0
            consol_volume   = consol_info['breakout_volume']  or 0
            cap_category    = classify_mcap(mcap_cr)

            if fund_score >= 60 and tech_score >= 65 and best_setup != 'Watching':
                tier = f'TIER 1 — BUY NOW ({best_setup})'
            elif fund_score >= 60 and in_consol and pct_to_breakout > -5:
                tier = 'TIER 1 — BREAKOUT IMMINENT'
            elif fund_score >= 60 and tech_score >= 40 and best_setup != 'Watching':
                tier = 'TIER 2 — WATCHLIST'
            elif fund_score >= 60 and in_consol:
                tier = 'TIER 2 — BASE BUILDING'
            else:
                tier = 'TIER 3 — WAITING'

            tech_reports.append({
                'Symbol'          : symbol,
                'Sector'          : sector,
                'Fund Score'      : fund_score,
                'Market Cap Cr'   : round(mcap_cr, 2),
                'Cap Category'    : cap_category,
                'Current Price'   : round(close,   2),
                'RSI'             : rsi,
                'ADX'             : adx,
                'MACD Hist'       : hist,
                'Vol Ratio'       : vol_r,
                'EMA20'           : ema20,
                'EMA50'           : ema50,
                'EMA200'          : ema200,
                'DI+'             : di_plus,
                'DI-'             : di_minus,
                'Momentum Score'  : mom_score,
                'Reversal Score'  : rev_score,
                'Best Setup'      : best_setup,
                'Tech Score'      : tech_score,
                'Tier'            : tier,
                'In Consolidation': in_consol,
                'Consol Days'     : consol_days,
                'Consol Label'    : consol_label,
                'Pct to Breakout' : round(pct_to_breakout, 2),
                'Consol Volume'   : consol_volume,
            })
            done_syms.add(symbol)

        except Exception as e:
            pass

        if (i + 1) % 50 == 0 or (i + 1) == len(remaining):
            pct = (i + 1) / len(remaining) * 100
            print(f"  [{i+1:4d}/{len(remaining)}] {pct:5.1f}% | Done: {len(tech_reports)}")
            with open(TECH_CHECKPOINT, 'wb') as f:
                pickle.dump({'reports': tech_reports, 'done': done_syms}, f)

    # ── SAVE ─────────────────────────────────────────────────
    tech_df = pd.DataFrame(tech_reports)
    tech_df.to_csv(TECH_OUTPUT, index=False)

    if os.path.exists(TECH_CHECKPOINT):
        os.remove(TECH_CHECKPOINT)

    print(f"\n✅ Technical analysis complete")
    print(f"   Stocks analysed : {len(tech_df)}")
    print(f"\nTier distribution:")
    print(tech_df['Tier'].value_counts().to_string())
    print(f"\nCap distribution:")
    print(tech_df['Cap Category'].value_counts().to_string())

Stocks to analyse : 752
Calculating indicators...
✅ Indicators done: 752 stocks
Remaining : 752 stocks
------------------------------------------------------------
  [  50/752]   6.6% | Done: 50
  [ 100/752]  13.3% | Done: 100
  [ 150/752]  19.9% | Done: 150
  [ 200/752]  26.6% | Done: 200
  [ 250/752]  33.2% | Done: 250
  [ 300/752]  39.9% | Done: 300
  [ 350/752]  46.5% | Done: 350
  [ 400/752]  53.2% | Done: 400
  [ 450/752]  59.8% | Done: 450
  [ 500/752]  66.5% | Done: 500
  [ 550/752]  73.1% | Done: 550
  [ 600/752]  79.8% | Done: 600
  [ 650/752]  86.4% | Done: 650
  [ 700/752]  93.1% | Done: 700
  [ 750/752]  99.7% | Done: 750
  [ 752/752] 100.0% | Done: 752

✅ Technical analysis complete
   Stocks analysed : 752

Tier distribution:
Tier
TIER 3 — WAITING               520
TIER 1 — BUY NOW (Reversal)     89
TIER 2 — WATCHLIST              87
TIER 2 — BASE BUILDING          32
TIER 1 — BUY NOW (Momentum)     22
TIER 1 — BREAKOUT IMMINENT       2

Cap distribution:
Cap Category
La

In [62]:
# ── QUICK SANITY CHECK ───────────────────────────────────────
tech_df = pd.read_csv('../data/technical_report_full.csv')

print("Top 10 Tier 1 stocks:")
tier1 = tech_df[tech_df['Tier'].str.contains('TIER 1')].sort_values('Fund Score', ascending=False)
print(tier1[['Symbol', 'Sector', 'Fund Score', 'Tech Score', 'Tier', 'Cap Category']].head(10).to_string(index=False))

print(f"\nTop 10 Tier 2 Watchlist:")
tier2 = tech_df[tech_df['Tier'] == 'TIER 2 — WATCHLIST'].sort_values('Fund Score', ascending=False)
print(tier2[['Symbol', 'Sector', 'Fund Score', 'Tech Score', 'Cap Category']].head(10).to_string(index=False))

print(f"\nLong base watch (top 10):")
consol = tech_df[tech_df['In Consolidation'] == True].sort_values('Consol Days', ascending=False)
print(consol[['Symbol', 'Fund Score', 'Consol Days', 'Cap Category']].head(10).to_string(index=False))

Top 10 Tier 1 stocks:
    Symbol                         Sector  Fund Score  Tech Score                        Tier   Cap Category
       BLS              Consumer Services        89.0          88 TIER 1 — BUY NOW (Reversal) Mini Large Cap
ALKYLAMINE                      Chemicals        87.0          70 TIER 1 — BUY NOW (Reversal) Mini Large Cap
 INDIAMART              Consumer Services        85.6          65 TIER 1 — BUY NOW (Reversal) Mini Large Cap
       FCL                      Chemicals        82.9          71 TIER 1 — BUY NOW (Reversal)        Mid Cap
     AGIIL                         Realty        81.0          65 TIER 1 — BUY NOW (Momentum)        Mid Cap
 LUMAXTECH Automobile and Auto Components        80.0          95 TIER 1 — BUY NOW (Momentum) Mini Large Cap
    ECLERX                       Services        80.0          73 TIER 1 — BUY NOW (Reversal) Mini Large Cap
      SBCL                  Capital Goods        79.0          80 TIER 1 — BUY NOW (Reversal)        Mid C

In [67]:
# ── CELL 13: WEEKLY REPORT DATA PREP ─────────────────────────

import yfinance as yf
from datetime import datetime

# ── LOAD DATA ─────────────────────────────────────────────────
with open('../data/price_data_full.pkl', 'rb') as f:
    price_data_full = pickle.load(f)

with open('../data/indicator_data_full.pkl', 'rb') as f:
    indicator_data_full = pickle.load(f)

tech_df    = pd.read_csv('../data/technical_report_full.csv')
fund_full  = pd.read_csv('../data/fundamental_scores_full.csv')
prefilt_df = pd.read_csv('../data/prefilt_passed.csv')

print(f"tech_df columns: {tech_df.columns.tolist()}")

# ── ADD SECTOR SCORE + CAP SCORE ─────────────────────────────
fund_full = fund_full.merge(
    prefilt_df[['Symbol', 'Market_Cap_Cr']], on='Symbol', how='left'
)

def classify_mcap(mcap_cr):
    if mcap_cr >= 20000:  return 'Large Cap'
    elif mcap_cr >= 5000: return 'Mini Large Cap'
    elif mcap_cr >= 1000: return 'Mid Cap'
    else:                 return 'Small Cap'

fund_full['Cap Category'] = fund_full['Market_Cap_Cr'].apply(
    lambda x: classify_mcap(x or 0)
)

fund_full['Sector Score'] = 0.0
for sector in fund_full['Sector'].unique():
    mask      = fund_full['Sector'] == sector
    max_score = fund_full[mask]['Final Score'].max()
    if max_score > 0:
        fund_full.loc[mask, 'Sector Score'] = (
            fund_full[mask]['Final Score'] / max_score * 10
        ).round(1)

fund_full['Cap Score'] = 0.0
for sector in fund_full['Sector'].unique():
    for cap in ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']:
        mask   = (fund_full['Sector'] == sector) & (fund_full['Cap Category'] == cap)
        subset = fund_full[mask]
        if len(subset) == 0:
            continue
        max_score = subset['Final Score'].max()
        if max_score > 0:
            fund_full.loc[mask, 'Cap Score'] = (
                subset['Final Score'] / max_score * 10
            ).round(1)

# ── DROP OLD SCORE COLUMNS IF EXIST + MERGE FRESH ────────────
for col in ['Sector Score', 'Cap Score', 'Sector Score_x', 'Cap Score_x',
            'Sector Score_y', 'Cap Score_y']:
    if col in tech_df.columns:
        tech_df = tech_df.drop(columns=[col])

tech_df = tech_df.merge(
    fund_full[['Symbol', 'Sector Score', 'Cap Score']],
    on='Symbol', how='left'
)
print(f"✅ Scores merged — Sector Score: {tech_df['Sector Score'].notna().sum()} stocks")

# ── RANK JUMPERS ──────────────────────────────────────────────
SCORES_FILE  = '../data/last_week_scores.csv'
rank_jumpers = set()

current_scores = tech_df[['Symbol', 'Sector Score', 'Cap Score']].copy()

if os.path.exists(SCORES_FILE):
    last_scores   = pd.read_csv(SCORES_FILE)
    merged_scores = current_scores.merge(
        last_scores[['Symbol', 'Sector Score', 'Cap Score']],
        on='Symbol', how='left', suffixes=('_now', '_prev')
    )
    for _, row in merged_scores.iterrows():
        prev_sec = row.get('Sector Score_prev', 0) or 0
        prev_cap = row.get('Cap Score_prev',    0) or 0
        curr_sec = row.get('Sector Score_now',  0) or 0
        curr_cap = row.get('Cap Score_now',     0) or 0
        sec_jump = (curr_sec - prev_sec) / prev_sec * 100 if prev_sec > 0 else 0
        cap_jump = (curr_cap - prev_cap) / prev_cap * 100 if prev_cap > 0 else 0
        if sec_jump >= 10 or cap_jump >= 10:
            rank_jumpers.add(row['Symbol'])
    print(f"✅ Rank jumpers found: {len(rank_jumpers)}")
else:
    print(f"ℹ️  No previous scores — jumpers will show from next week")

current_scores.to_csv(SCORES_FILE, index=False)
print(f"✅ Scores saved for next week")

# ── VOL 5D RATIO ──────────────────────────────────────────────
vol_5d_data = {}
for symbol, df in price_data_full.items():
    try:
        if len(df) < 20:
            vol_5d_data[symbol] = 1.0
            continue
        vol_5d_avg      = df['Volume'].iloc[-5:].mean()
        vol_prior15_avg = df['Volume'].iloc[-20:-5].mean()
        vol_5d_data[symbol] = round(
            vol_5d_avg / vol_prior15_avg if vol_prior15_avg > 0 else 1.0, 2
        )
    except:
        vol_5d_data[symbol] = 1.0

tech_df['Vol 5D Ratio'] = tech_df['Symbol'].map(vol_5d_data).fillna(1.0)
print(f"✅ Vol 5D Ratio done")

# ── BREAKOUT VOLUME ───────────────────────────────────────────
breakout_vol_data = {}
for _, row in tech_df[tech_df['In Consolidation'] == True].iterrows():
    symbol      = row['Symbol']
    consol_days = int(row['Consol Days'])
    if symbol not in price_data_full:
        continue
    df = price_data_full[symbol]
    if len(df) < consol_days + 5:
        breakout_vol_data[symbol] = 1.0
        continue
    week_vol   = df['Volume'].iloc[-5:].mean()
    consol_avg = df['Volume'].iloc[-consol_days:-5].mean()
    breakout_vol_data[symbol] = round(
        week_vol / consol_avg if consol_avg > 0 else 1.0, 2
    )

tech_df['Breakout Vol'] = tech_df['Symbol'].map(breakout_vol_data).fillna(0.0)
print(f"✅ Breakout Vol done")

# ── VOLUME INFERENCE ──────────────────────────────────────────
def get_volume_inference(row):
    vol   = float(row.get('Vol 5D Ratio', 1.0))
    setup = str(row['Best Setup'])
    adx   = float(row['ADX'])
    if   vol >= 3.0: vol_label = 'Extremely High'
    elif vol >= 2.0: vol_label = 'Very High'
    elif vol >= 1.5: vol_label = 'High'
    elif vol >= 1.0: vol_label = 'Normal'
    elif vol >= 0.7: vol_label = 'Low'
    else:            vol_label = 'Very Low'
    if setup == 'Momentum':
        if   vol >= 2.0: inf = 'Strong participation → trend likely to continue'
        elif vol >= 1.5: inf = 'Good volume support → momentum confirmed'
        elif vol >= 1.0: inf = 'Average volume → monitor for increase'
        elif vol >= 0.7: inf = 'Weak volume → momentum may fade, wait'
        else:            inf = 'Very low volume → no conviction, caution'
    elif setup == 'Reversal':
        if   vol <= 0.5: inf = 'Sellers exhausted → reversal likely'
        elif vol <= 0.7: inf = 'Low volume on down move → selling pressure easing'
        elif vol <= 1.0: inf = 'Neutral volume → wait for low-volume base to form'
        elif vol <= 1.5: inf = 'Elevated volume on reversal → watch direction'
        else:            inf = 'High volume → could be capitulation or distribution'
    else:
        if   adx > 40 and vol >= 2.0: inf = 'Above avg volume but no setup → watch closely'
        elif adx > 40:                inf = 'Strong downtrend → no entry signal'
        elif vol <= 0.5:              inf = 'Very low volume → no interest, wait'
        elif vol >= 1.5 and adx < 35: inf = 'Above avg volume but no setup → watch'
        elif vol >= 1.0:              inf = 'Normal activity → no setup forming yet'
        else:                         inf = 'Low activity → no setup forming yet'
    return vol_label, inf

for idx, row in tech_df.iterrows():
    label, inf = get_volume_inference(row)
    tech_df.at[idx, 'Vol Label']     = label
    tech_df.at[idx, 'Vol Inference'] = inf
print(f"✅ Volume inference done")

# ── SECTOR TREND ──────────────────────────────────────────────
SECTOR_INDEX_MAP = {
    'Information Technology'            : '^CNXIT',
    'Healthcare'                        : '^CNXPHARMA',
    'Financial Services'                : 'FINIETF.NS',
    'Capital Goods'                     : '^CNXINFRA',
    'Chemicals'                         : 'MID150BEES.NS',
    'Consumer Durables'                 : '^CNXCONSUM',
    'Oil, Gas & Consumable Fuels'       : '^CNXENERGY',
    'Automobile and Auto Components'    : '^CNXAUTO',
    'Textiles'                          : 'MID150BEES.NS',
    'Consumer Services'                 : 'MID150BEES.NS',
    'Banking'                           : '^NSEBANK',
    'Fast Moving Consumer Goods'        : '^CNXFMCG',
    'Metals & Mining'                   : '^CNXMETAL',
    'Realty'                            : '^CNXREALTY',
    'Services'                          : '^CNXSERVICE',
    'Construction'                      : '^CNXINFRA',
    'Power'                             : '^CNXENERGY',
    'Pharmaceuticals'                   : '^CNXPHARMA',
    'Cement'                            : '^CNXCMDT',
    'Defence'                           : 'MID150BEES.NS',
    'Telecommunication'                 : 'MID150BEES.NS',
    'Diversified'                       : 'MID150BEES.NS',
    'Utilities'                         : '^CNXENERGY',
    'Media Entertainment & Publication' : 'MID150BEES.NS',
    'Agriculture'                       : 'MID150BEES.NS',
    'Forest Materials'                  : 'MID150BEES.NS',
}

print("Fetching sector data...")
ticker_data = {}
for ticker in set(SECTOR_INDEX_MAP.values()):
    try:
        df = yf.download(ticker, period='1y', interval='1d',
                         progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [col[0] for col in df.columns]
        if len(df) > 50:
            ticker_data[ticker] = df
    except:
        pass

sector_price = {}
fallback     = 'MID150BEES.NS'
for sector, ticker in SECTOR_INDEX_MAP.items():
    if ticker in ticker_data:
        sector_price[sector] = ticker_data[ticker]
    elif fallback in ticker_data:
        sector_price[sector] = ticker_data[fallback]

def calculate_sector_indicators(df):
    data = df.copy()
    data['EMA20']  = data['Close'].ewm(span=20,  adjust=False).mean()
    data['EMA50']  = data['Close'].ewm(span=50,  adjust=False).mean()
    data['EMA200'] = data['Close'].ewm(span=200, adjust=False).mean()
    delta    = data['Close'].diff()
    gain     = delta.where(delta > 0, 0)
    loss     = -delta.where(delta < 0, 0)
    rs       = gain.ewm(span=14, adjust=False).mean() / (
                loss.ewm(span=14, adjust=False).mean() + 1e-9)
    data['RSI']       = 100 - (100 / (1 + rs))
    ema12             = data['Close'].ewm(span=12, adjust=False).mean()
    ema26             = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD']      = ema12 - ema26
    data['Signal']    = data['MACD'].ewm(span=9, adjust=False).mean()
    data['MACD_Hist'] = data['MACD'] - data['Signal']
    high     = data['High']
    low      = data['Low']
    close    = data['Close']
    tr       = pd.concat([high - low, (high - close.shift()).abs(),
                          (low - close.shift()).abs()], axis=1).max(axis=1)
    dm_plus  = high.diff()
    dm_minus = -low.diff()
    dm_plus  = dm_plus.where((dm_plus > dm_minus) & (dm_plus > 0), 0)
    dm_minus = dm_minus.where((dm_minus > dm_plus) & (dm_minus > 0), 0)
    atr      = tr.ewm(span=14, adjust=False).mean()
    di_plus  = 100 * dm_plus.ewm(span=14, adjust=False).mean() / (atr + 1e-9)
    di_minus = 100 * dm_minus.ewm(span=14, adjust=False).mean() / (atr + 1e-9)
    dx       = 100 * (di_plus - di_minus).abs() / (di_plus + di_minus + 1e-9)
    data['ADX']      = dx.ewm(span=14, adjust=False).mean()
    data['DI_Plus']  = di_plus
    data['DI_Minus'] = di_minus
    return data

def get_sector_trend(sector):
    if sector not in sector_price:
        return 'No data', '—'
    df     = calculate_sector_indicators(sector_price[sector])
    latest = df.iloc[-1]
    close    = latest['Close']
    rsi      = round(latest['RSI'],   1)
    adx      = round(latest['ADX'],   1)
    ema20    = latest['EMA20']
    ema50    = latest['EMA50']
    ema200   = latest['EMA200']
    di_plus  = latest['DI_Plus']
    di_minus = latest['DI_Minus']
    macd     = latest['MACD']
    signal   = latest['Signal']
    macd_h   = latest['MACD_Hist']
    above_ema200 = close > ema200
    above_ema50  = close > ema50
    ema_aligned  = ema20 > ema50 > ema200
    ema_bearish  = ema20 < ema50 < ema200
    uptrend      = di_plus > di_minus
    macd_bull    = macd > signal and macd_h > 0
    macd_bear    = macd < signal and macd_h < 0
    bull_score   = sum([above_ema200, above_ema50, ema_aligned,
                        uptrend, macd_bull, rsi > 55])
    bear_score   = sum([not above_ema200, not above_ema50, ema_bearish,
                        not uptrend, macd_bear, rsi < 45])
    if   bull_score >= 5 and adx > 25:  label = 'Strong Uptrend ↑↑'
    elif bull_score >= 4 and adx > 20:  label = 'Uptrend ↑'
    elif bull_score >= 3 and adx <= 20: label = 'Weak Uptrend →↑'
    elif bull_score == bear_score:      label = 'Sideways →'
    elif bear_score >= 5 and adx > 25:  label = 'Strong Downtrend ↓↓'
    elif bear_score >= 4 and adx > 20:  label = 'Downtrend ↓'
    elif bear_score >= 3 and adx <= 20: label = 'Weak Downtrend →↓'
    else:                               label = 'Sideways →'
    ema_str  = '↑↑↑' if ema_aligned else '↓↓↓' if ema_bearish else 'mixed'
    macd_str = '▲' if macd_bull else '▼' if macd_bear else '~'
    detail   = (f"RSI {rsi} | ADX {adx} | MACD {macd_str} | "
                f"EMA {ema_str} | {'Above' if above_ema200 else 'Below'} EMA200")
    return label, detail

for idx, row in tech_df.iterrows():
    s_label, s_detail = get_sector_trend(str(row['Sector']))
    tech_df.at[idx, 'Sector Trend']  = s_label
    tech_df.at[idx, 'Sector Detail'] = s_detail
print(f"✅ Sector trends done")

# ── SAVE ─────────────────────────────────────────────────────
tech_df.to_csv('../data/technical_report_full.csv', index=False)
print(f"✅ Saved: technical_report_full.csv")
print(f"\nColumns now: {tech_df.columns.tolist()}")
print(f"\nReady for Cell 14")

tech_df columns: ['Symbol', 'Sector', 'Fund Score', 'Market Cap Cr', 'Cap Category', 'Current Price', 'RSI', 'ADX', 'MACD Hist', 'Vol Ratio', 'EMA20', 'EMA50', 'EMA200', 'DI+', 'DI-', 'Momentum Score', 'Reversal Score', 'Best Setup', 'Tech Score', 'Tier', 'In Consolidation', 'Consol Days', 'Consol Label', 'Pct to Breakout', 'Consol Volume', 'Sector Score', 'Cap Score', 'Vol 5D Ratio', 'Breakout Vol', 'range_pct', 'Vol Label', 'Vol Inference', 'Sector Trend', 'Sector Detail']
✅ Scores merged — Sector Score: 752 stocks
✅ Rank jumpers found: 0
✅ Scores saved for next week
✅ Vol 5D Ratio done
✅ Breakout Vol done
✅ Volume inference done
Fetching sector data...
✅ Sector trends done
✅ Saved: technical_report_full.csv

Columns now: ['Symbol', 'Sector', 'Fund Score', 'Market Cap Cr', 'Cap Category', 'Current Price', 'RSI', 'ADX', 'MACD Hist', 'Vol Ratio', 'EMA20', 'EMA50', 'EMA200', 'DI+', 'DI-', 'Momentum Score', 'Reversal Score', 'Best Setup', 'Tech Score', 'Tier', 'In Consolidation', 'Consol

In [69]:
# ── CELL 14: PRINT WEEKLY REPORT ─────────────────────────────

tech_df = pd.read_csv('../data/technical_report_full.csv')

# ── REPORT MODE ───────────────────────────────────────────────
print("Select report type:")
print("  1 = Short Report")
print("  2 = Long Report")
print("  3 = Both")
report_mode = input("Enter 1, 2, or 3: ").strip()
if report_mode not in ['1', '2', '3']:
    report_mode = '2'
    print("Invalid input — defaulting to Long Report")

GENERATE_SHORT = report_mode in ['1', '3']
GENERATE_LONG  = report_mode in ['2', '3']

CAP_ORDER = ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']

# ── BREAKOUT TRACKER ──────────────────────────────────────────
BREAKOUT_FILE = '../data/breakout_tracker.csv'
today_str     = datetime.now().strftime('%Y-%m-%d')

# Load existing tracker
if os.path.exists(BREAKOUT_FILE):
    bk_df = pd.read_csv(BREAKOUT_FILE)
else:
    bk_df = pd.DataFrame(columns=[
        'Symbol', 'First_Breakout_Date', 'Weeks_Count',
        'Last_Pct', 'Cap_Category', 'Sector', 'Fund_Score'
    ])

# Current breakouts — stocks above range high (Pct to Breakout > 0)
current_breakouts = tech_df[
    (tech_df['In Consolidation'] == True) &
    (tech_df['Pct to Breakout'] > 0)
][['Symbol', 'Pct to Breakout', 'Cap Category',
   'Sector', 'Fund Score', 'Current Price',
   'Vol 5D Ratio', 'RSI', 'Tech Score']].copy()

# Update tracker
updated_rows = []
for _, row in current_breakouts.iterrows():
    sym = row['Symbol']
    existing = bk_df[bk_df['Symbol'] == sym]
    if len(existing) > 0:
        weeks = int(existing.iloc[0]['Weeks_Count']) + 1
        updated_rows.append({
            'Symbol'             : sym,
            'First_Breakout_Date': existing.iloc[0]['First_Breakout_Date'],
            'Weeks_Count'        : weeks,
            'Last_Pct'           : round(row['Pct to Breakout'], 2),
            'Cap_Category'       : row['Cap Category'],
            'Sector'             : row['Sector'],
            'Fund_Score'         : row['Fund Score'],
        })
    else:
        updated_rows.append({
            'Symbol'             : sym,
            'First_Breakout_Date': today_str,
            'Weeks_Count'        : 1,
            'Last_Pct'           : round(row['Pct to Breakout'], 2),
            'Cap_Category'       : row['Cap Category'],
            'Sector'             : row['Sector'],
            'Fund_Score'         : row['Fund Score'],
        })

# Keep stocks from last 3 weeks even if not currently breaking out
# (they might have pulled back slightly)
for _, row in bk_df.iterrows():
    sym = row['Symbol']
    already_added = any(r['Symbol'] == sym for r in updated_rows)
    if not already_added and int(row['Weeks_Count']) < 3:
        # Check if still near breakout (within -5%)
        tech_row = tech_df[tech_df['Symbol'] == sym]
        if len(tech_row) > 0:
            pct = float(tech_row.iloc[0]['Pct to Breakout'])
            if pct > -5:  # still near or above breakout
                updated_rows.append({
                    'Symbol'             : sym,
                    'First_Breakout_Date': row['First_Breakout_Date'],
                    'Weeks_Count'        : int(row['Weeks_Count']),
                    'Last_Pct'           : round(pct, 2),
                    'Cap_Category'       : row['Cap_Category'],
                    'Sector'             : row['Sector'],
                    'Fund_Score'         : row['Fund_Score'],
                })

new_bk_df = pd.DataFrame(updated_rows)
new_bk_df.to_csv(BREAKOUT_FILE, index=False)
print(f"✅ Breakout tracker updated: {len(new_bk_df)} stocks")

# Merge breakout info into tech_df for display
if len(new_bk_df) > 0:
    tech_df = tech_df.merge(
        new_bk_df[['Symbol', 'Weeks_Count', 'First_Breakout_Date']],
        on='Symbol', how='left'
    )
else:
    tech_df['Weeks_Count']        = None
    tech_df['First_Breakout_Date'] = None

# ── HELPER FUNCTIONS ──────────────────────────────────────────
def get_reason(row):
    setup  = str(row['Best Setup'])
    rsi    = float(row['RSI'])
    adx    = float(row['ADX'])
    macd   = float(row['MACD Hist'])
    price  = float(row['Current Price'])
    ema20  = float(row['EMA20'])
    ema50  = float(row['EMA50'])
    ema200 = float(row['EMA200'])
    above_ema200 = price > ema200
    ema_aligned  = ema20 > ema50 > ema200
    if setup == 'Momentum':
        if ema_aligned and rsi > 55:      return f'Strong uptrend, EMA aligned, RSI {rsi:.0f}'
        elif above_ema200 and rsi > 50:   return f'Momentum building, above EMA200, RSI {rsi:.0f}'
        elif above_ema200 and rsi <= 50:  return f'Consolidating above EMA200, RSI {rsi:.0f}'
        elif not above_ema200 and rsi>60: return f'Short bounce, below EMA200, RSI {rsi:.0f}'
        else:                             return f'Weak momentum, below EMA200, RSI {rsi:.0f}'
    elif setup == 'Reversal':
        if rsi < 30:   return f'Deeply oversold RSI {rsi:.0f}, reversal imminent'
        elif rsi < 35: return f'Oversold RSI {rsi:.0f}, watch for turn'
        elif macd > 0: return f'RSI {rsi:.0f}, MACD turning up'
        else:          return f'Divergence forming, RSI {rsi:.0f}'
    else:
        if adx > 40:   return f'Strong downtrend ADX {adx:.0f}, wait'
        elif adx > 25: return f'Downtrend ADX {adx:.0f}, no setup yet'
        elif rsi < 30: return f'Deeply oversold RSI {rsi:.0f}, watch for turn'
        elif rsi < 40: return f'Oversold RSI {rsi:.0f}, watch for turn'
        else:          return f'RSI {rsi:.0f}, setup not confirmed'

def get_breakout_vol_inference(week_vol, break_vol):
    if   break_vol >= 2.0: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡⚡ → Strong breakout volume building"
    elif break_vol >= 1.5: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ⚡  → Volume building toward breakout"
    elif break_vol >= 1.0: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x ~  → Normal volume, no breakout pressure"
    elif break_vol >= 0.7: return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x    → Quiet, below consolidation avg"
    else:                  return f"Week Vol:{week_vol:.2f}x | Break Vol:{break_vol:.2f}x    → Very quiet, no breakout interest"

def passes_rank_filter(row):
    return float(row.get('Cap Score', 0) or 0) >= 7 or \
           float(row.get('Sector Score', 0) or 0) >= 7

def is_rank_jumper(row):
    return row['Symbol'] in rank_jumpers

def cap_order(x):
    return CAP_ORDER.index(x) if x in CAP_ORDER else 99

def mcap_str(mcap_cr):
    try:
        v = float(mcap_cr)
        if v >= 100000: return f"Rs{v/100000:.1f}L Cr"
        return f"Rs{v:,.0f}Cr"
    except:
        return "Rs—"

# ── PREPARE DATA SLICES ───────────────────────────────────────
# Tier 1 Momentum
tier1_mom = tech_df[
    tech_df['Tier'].str.startswith('TIER 1') &
    (tech_df['Best Setup'] == 'Momentum')
].copy()
tier1_mom['_cap_order'] = tier1_mom['Cap Category'].apply(cap_order)
tier1_mom = tier1_mom.sort_values(['_cap_order', 'Fund Score'],
                                   ascending=[True, False]).reset_index(drop=True)

# Tier 2A — rank filtered, top 25
tier2a = tech_df[tech_df['Tier'] == 'TIER 2 — WATCHLIST'].copy()
tier2a = tier2a[
    tier2a.apply(passes_rank_filter, axis=1) |
    tier2a['Symbol'].isin(rank_jumpers)
].sort_values('Fund Score', ascending=False).head(25).reset_index(drop=True)

# Tier 2B — rank filtered, top 25
tier2b = tech_df[tech_df['Tier'] == 'TIER 2 — BASE BUILDING'].copy()
tier2b = tier2b[
    tier2b.apply(passes_rank_filter, axis=1) |
    tier2b['Symbol'].isin(rank_jumpers)
].sort_values('Fund Score', ascending=False).head(25).reset_index(drop=True)

# Tier 3 Momentum — rank filtered
tier3_mom = tech_df[
    (tech_df['Tier'] == 'TIER 3 — WAITING') &
    (tech_df['Best Setup'] == 'Momentum')
].copy()
tier3_mom['_cap_order'] = tier3_mom['Cap Category'].apply(cap_order)
tier3_mom = tier3_mom[
    tier3_mom.apply(passes_rank_filter, axis=1) |
    tier3_mom['Symbol'].isin(rank_jumpers)
].sort_values(['_cap_order', 'Fund Score'],
               ascending=[True, False]).reset_index(drop=True)

# Near breakout (within 10%)
consol_near = tech_df[
    (tech_df['In Consolidation'] == True) &
    (tech_df['Pct to Breakout'] >= -10) &
    (tech_df['Pct to Breakout'] <= 0)   # not yet broken out
].sort_values('Pct to Breakout', ascending=False).reset_index(drop=True)

# Breakout confirmed stocks (from tracker)
breakout_confirmed = tech_df[
    tech_df['Weeks_Count'].notna() &
    (tech_df['Pct to Breakout'] > 0)
].copy()
if len(breakout_confirmed) > 0:
    breakout_confirmed['_cap_order'] = breakout_confirmed['Cap Category'].apply(cap_order)
    breakout_confirmed = breakout_confirmed.sort_values(
        ['_cap_order', 'Pct to Breakout'], ascending=[True, False])

# Reversal candidates — top 10 per cap
all_reversal = tech_df[tech_df['Best Setup'] == 'Reversal'].copy()
reversal_by_cap = {}
for cap in CAP_ORDER:
    subset = all_reversal[all_reversal['Cap Category'] == cap].copy()
    reversal_by_cap[cap] = subset.sort_values(
        'Reversal Score', ascending=False).head(10)

# ── GENERATE REPORT ───────────────────────────────────────────
def generate_report(is_short):
    lines  = []
    serial = 1

    def p(line=''):
        lines.append(str(line))
        print(line)

    mode_label = 'SHORT' if is_short else 'LONG'
    today      = datetime.now().strftime('%d %B %Y')

    p(f"{'='*78}")
    p(f"  AI STOCK SCREENER — WEEKLY REPORT ({mode_label})")
    p(f"  {today}  |  Universe: {len(tech_df)} stocks")
    p(f"{'='*78}")
    p(f"  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + cap")
    p(f"  Week Vol = this week avg vs prior 15-day avg")
    p(f"  Break Vol= this week avg vs consolidation period avg")
    p(f"  ↑ = jumped 10%+ in rank vs last week")

    # ── SECTOR SUMMARY (long only) ────────────────────────────
    if not is_short:
        p(f"\n{'─'*78}")
        p(f"  SECTOR TREND SUMMARY")
        p(f"{'─'*78}")
        p(f"  {'Sector':42} {'Trend':28} Detail")
        p(f"  {'─'*76}")
        for sector in sorted(SECTOR_INDEX_MAP.keys()):
            label, detail = get_sector_trend(sector)
            p(f"  {sector:42} {label:28} {detail}")

    # ── TIER 1 MOMENTUM ───────────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  TIER 1 — MOMENTUM  ({len(tier1_mom)} stocks)")
    p(f"  Fund ≥60 + Tech ≥65 + Momentum setup confirmed")
    p(f"{'─'*78}")

    current_cap = None
    for _, row in tier1_mom.iterrows():
        cap = row['Cap Category']
        if cap != current_cap:
            current_cap = cap
            cap_short   = {'Large Cap':'L','Mini Large Cap':'ML',
                           'Mid Cap':'M','Small Cap':'S'}.get(cap,'?')
            p(f"\n  [{cap_short}] {cap}")
            p(f"  {'─'*68}")
        consol_line = ''
        if row['In Consolidation'] and row['Consol Days'] > 0:
            bvol_inf    = get_breakout_vol_inference(
                row['Vol 5D Ratio'], row['Breakout Vol'])
            consol_line = (f"\n    Base    : {row['Consol Days']}d "
                           f"({row['Consol Label']}) | "
                           f"{row['Pct to Breakout']:+.1f}% to breakout"
                           f"\n    Break   : {bvol_inf}")
        jump_tag = ' ↑' if is_rank_jumper(row) else ''
        p(f"""
  #{serial}  {row['Symbol']}{jump_tag}  ({row['Sector']})  MCap {mcap_str(row['Market Cap Cr'])}  Rs{row['Current Price']:.2f}
    Setup   : {row['Best Setup']:10}  Tech {row['Tech Score']:3.0f}/100
    Scores  : Fund {row['Fund Score']}/100  Sector Rank {row['Sector Score']}/10  Cap Rank {row['Cap Score']}/10{consol_line}
    Tech    : RSI {row['RSI']:.0f}  ADX {row['ADX']:.0f}  MACD {row['MACD Hist']:+.2f}  → {get_reason(row)}
    Volume  : Week Vol:{row['Vol 5D Ratio']:.2f}x ({row['Vol Label']}) → {row['Vol Inference']}
    Sector  : {row['Sector Trend']} | {row['Sector Detail']}""")
        serial += 1

    # ── TIER 2A ───────────────────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  TIER 2A — WATCHLIST  (Top 25 | Cap/Sec Rank ≥7 or ↑)")
    p(f"  Setup forming — wait for confirmation before entering")
    p(f"{'─'*78}")

    for _, row in tier2a.iterrows():
        consol_tag = f"  [{row['Consol Days']}d base]" if row['In Consolidation'] else ''
        jump_tag   = ' ↑' if is_rank_jumper(row) else ''
        p(f"\n  #{serial}  {row['Symbol']}{jump_tag}  "
          f"Rs{row['Current Price']:.2f}  "
          f"MCap {mcap_str(row['Market Cap Cr'])}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}{consol_tag}")
        p(f"      Technical : [{row['Best Setup']}|{row['Tech Score']:.0f}]  "
          f"RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
        p(f"      Volume    : Week Vol:{row['Vol 5D Ratio']:.2f}x "
          f"({row['Vol Label']}) → {row['Vol Inference']}")
        if not is_short:
            p(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
        serial += 1

    # ── TIER 2B ───────────────────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  TIER 2B — BASE BUILDING  (Top 25 | Cap/Sec Rank ≥7 or ↑)")
    p(f"  Long consolidation base — watch for volume breakout")
    p(f"{'─'*78}")

    for _, row in tier2b.iterrows():
        bvol_inf = get_breakout_vol_inference(row['Vol 5D Ratio'], row['Breakout Vol'])
        jump_tag = ' ↑' if is_rank_jumper(row) else ''
        p(f"\n  #{serial}  {row['Symbol']}{jump_tag}  "
          f"Rs{row['Current Price']:.2f}  "
          f"MCap {mcap_str(row['Market Cap Cr'])}  "
          f"Fund:{row['Fund Score']:4.1f}  "
          f"Sec:{row['Sector Score']:4.1f}  "
          f"Cap:{row['Cap Score']:4.1f}  "
          f"{row['Consol Days']}d  "
          f"ToBreak:{row['Pct to Breakout']:+.1f}%")
        p(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
          f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
        p(f"      Volume    : {bvol_inf}")
        if not is_short:
            p(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
        serial += 1

    # ── TIER 3 MOMENTUM (long only) ───────────────────────────
    if not is_short:
        p(f"\n{'─'*78}")
        p(f"  TIER 3 — MOMENTUM WAITING  "
          f"({len(tier3_mom)} shown | Cap/Sec Rank ≥7 or ↑)")
        p(f"  Good businesses with momentum — setup not yet confirmed")
        p(f"{'─'*78}")

        current_cap = None
        for _, row in tier3_mom.iterrows():
            cap = row['Cap Category']
            if cap != current_cap:
                current_cap = cap
                cap_short   = {'Large Cap':'L','Mini Large Cap':'ML',
                               'Mid Cap':'M','Small Cap':'S'}.get(cap,'?')
                p(f"\n  [{cap_short}] {cap}")
                p(f"  {'─'*68}")
            jump_tag = ' ↑' if is_rank_jumper(row) else ''
            if row['In Consolidation'] and row['Breakout Vol'] > 0:
                vol_line = get_breakout_vol_inference(
                    row['Vol 5D Ratio'], row['Breakout Vol'])
            else:
                vol_line = (f"Week Vol:{row['Vol 5D Ratio']:.2f}x "
                           f"({row['Vol Label']}) → {row['Vol Inference']}")
            p(f"\n  #{serial}  {row['Symbol']}{jump_tag}  "
              f"Rs{row['Current Price']:.2f}  "
              f"MCap {mcap_str(row['Market Cap Cr'])}  "
              f"Fund:{row['Fund Score']:4.1f}  "
              f"Sec:{row['Sector Score']:4.1f}  "
              f"Cap:{row['Cap Score']:4.1f}")
            p(f"      Technical : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
              f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
            p(f"      Volume    : {vol_line}")
            p(f"      Sector    : {row['Sector Trend']} | {row['Sector Detail']}")
            serial += 1

    # ── BREAKOUT CONFIRMED ────────────────────────────────────
    if len(breakout_confirmed) > 0:
        p(f"\n{'─'*78}")
        p(f"  🚀 BREAKOUT CONFIRMED  ({len(breakout_confirmed)} stocks)")
        p(f"  Stocks that broke above consolidation range")
        p(f"  Shown for 3 weeks from breakout date")
        p(f"{'─'*78}")
        p(f"  {'Symbol':12} {'Price':>9} {'MCap':>12} {'Above%':>7} "
          f"{'Weeks':>6} {'WkVol':>7} {'RSI':>5}  Breakout Date")
        p(f"  {'─'*85}")

        current_cap = None
        for _, row in breakout_confirmed.iterrows():
            cap = row['Cap Category']
            if cap != current_cap:
                current_cap = cap
                cap_short   = {'Large Cap':'L','Mini Large Cap':'ML',
                               'Mid Cap':'M','Small Cap':'S'}.get(cap,'?')
                p(f"\n  [{cap_short}] {cap}")

            weeks    = int(row['Weeks_Count']) if pd.notna(row['Weeks_Count']) else 1
            bk_date  = row['First_Breakout_Date'] if pd.notna(row['First_Breakout_Date']) else today_str
            pct      = row['Pct to Breakout']
            wvol     = row['Vol 5D Ratio']
            wvol_tag = '⚡' if wvol >= 1.5 else ' '
            p(f"  {row['Symbol']:12} "
              f"Rs{row['Current Price']:>8.2f} "
              f"{mcap_str(row['Market Cap Cr']):>12} "
              f"{pct:>+6.1f}%  "
              f"Wk{weeks}/3  "
              f"{wvol:.2f}x{wvol_tag}  "
              f"RSI{row['RSI']:>3.0f}  "
              f"[{bk_date}]")

    # ── NEAR BREAKOUT ─────────────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  NEAR BREAKOUT BASE WATCH  "
      f"({len(consol_near)} stocks within 10% of breakout)")
    p(f"  Break Vol > 1.5x = volume building — high priority watch")
    p(f"{'─'*78}")
    p(f"  {'Symbol':12} {'Price':>9} {'MCap':>12} {'Days':>5} "
      f"{'ToBreak':>8} {'WkVol':>7} {'BrkVol':>7}  Status")
    p(f"  {'─'*85}")

    for _, row in consol_near.iterrows():
        pct      = row['Pct to Breakout']
        bvol     = row['Breakout Vol']
        wvol     = row['Vol 5D Ratio']
        jump_tag = ' ↑' if is_rank_jumper(row) else ''
        status   = 'NEAR BREAKOUT ⚡' if pct > -5 else 'Approaching'
        bvol_tag = '⚡' if bvol >= 1.5 else '~' if bvol >= 0.8 else ' '
        p(f"  {row['Symbol'] + jump_tag:12} "
          f"Rs{row['Current Price']:>8.2f} "
          f"{mcap_str(row['Market Cap Cr']):>12} "
          f"{row['Consol Days']:>5}d "
          f"{pct:>+7.1f}%  "
          f"{wvol:.2f}x  "
          f"{bvol:.2f}x{bvol_tag}  "
          f"[{status}]")

    # ── REVERSAL CANDIDATES ───────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  BULLISH REVERSAL CANDIDATES  (Top 10 per cap category)")
    p(f"  Oversold + divergence forming — wait for confirmation")
    p(f"{'─'*78}")

    for cap in CAP_ORDER:
        subset = reversal_by_cap.get(cap, pd.DataFrame())
        if len(subset) == 0:
            continue
        cap_short = {'Large Cap':'L','Mini Large Cap':'ML',
                     'Mid Cap':'M','Small Cap':'S'}.get(cap,'?')
        p(f"\n  [{cap_short}] {cap}")
        p(f"  {'─'*68}")
        for _, row in subset.iterrows():
            jump_tag   = ' ↑' if is_rank_jumper(row) else ''
            tier_short = (row['Tier']
                          .replace('TIER 1 — BUY NOW (Reversal)', 'T1-Rev')
                          .replace('TIER 1 — BUY NOW (Momentum)', 'T1-Mom')
                          .replace('TIER 2 — WATCHLIST',          'T2')
                          .replace('TIER 2 — BASE BUILDING',      'T2B')
                          .replace('TIER 3 — WAITING',            'T3'))
            p(f"\n  #{serial}  {row['Symbol']}{jump_tag}  "
              f"Rs{row['Current Price']:.2f}  "
              f"MCap {mcap_str(row['Market Cap Cr'])}  "
              f"[{tier_short}]  "
              f"Fund:{row['Fund Score']:4.1f}  "
              f"Rev:{row['Reversal Score']:.0f}  "
              f"Sec:{row['Sector Score']:4.1f}  Cap:{row['Cap Score']:4.1f}")
            p(f"      Tech    : RSI:{row['RSI']:3.0f}  ADX:{row['ADX']:3.0f}  "
              f"MACD:{round(row['MACD Hist']):+d}  → {get_reason(row)}")
            p(f"      Volume  : Week Vol:{row['Vol 5D Ratio']:.2f}x "
              f"({row['Vol Label']}) → {row['Vol Inference']}")
            if not is_short:
                p(f"      Sector  : {row['Sector Trend']} | {row['Sector Detail']}")
            serial += 1

    # ── RANK JUMPERS ──────────────────────────────────────────
    if rank_jumpers:
        p(f"\n{'─'*78}")
        p(f"  ↑ RANK JUMPERS THIS WEEK  ({len(rank_jumpers)} stocks)")
        p(f"{'─'*78}")
        jumper_df = tech_df[tech_df['Symbol'].isin(rank_jumpers)].sort_values(
            'Fund Score', ascending=False)
        for _, row in jumper_df.iterrows():
            p(f"  {row['Symbol']:12}  Rs{row['Current Price']:.2f}  "
              f"MCap {mcap_str(row['Market Cap Cr'])}  "
              f"Fund:{row['Fund Score']:4.1f}  "
              f"Sec:{row['Sector Score']:4.1f}  "
              f"Cap:{row['Cap Score']:4.1f}  "
              f"{row['Tier']}  {row['Sector']}")

    # ── FOOTER ────────────────────────────────────────────────
    p(f"\n{'─'*78}")
    p(f"  L=Large Cap  ML=Mini Large Cap  M=Mid Cap  S=Small Cap")
    p(f"  SecRank  = rank vs same sector | CapRank = rank vs same sector + cap")
    p(f"  Week Vol = this week avg vs prior 15-day avg")
    p(f"  Break Vol= this week avg vs consolidation period avg")
    p(f"  ↑ = jumped 10%+ in Cap or Sector rank vs last week")
    p(f"{'─'*78}")

    return lines

# ── RUN & SAVE ────────────────────────────────────────────────
weekly_dir = '../data/weekly_reports'
os.makedirs(weekly_dir, exist_ok=True)

if GENERATE_SHORT:
    print("\n" + "="*78)
    print("  GENERATING SHORT REPORT")
    print("="*78 + "\n")
    short_lines = generate_report(is_short=True)
    short_path  = f'{weekly_dir}/weekly_report_short_{today_str}.txt'
    with open(short_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(short_lines))
    print(f"\n✅ Short report saved: {short_path}")

if GENERATE_LONG:
    print("\n" + "="*78)
    print("  GENERATING LONG REPORT")
    print("="*78 + "\n")
    long_lines = generate_report(is_short=False)
    long_path  = f'{weekly_dir}/weekly_report_long_{today_str}.txt'
    with open(long_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(long_lines))
    print(f"\n✅ Long report saved: {long_path}")

Select report type:
  1 = Short Report
  2 = Long Report
  3 = Both


Enter 1, 2, or 3:  1


✅ Breakout tracker updated: 3 stocks

  GENERATING SHORT REPORT

  AI STOCK SCREENER — WEEKLY REPORT (SHORT)
  23 March 2026  |  Universe: 752 stocks
  SecRank  = rank vs same sector  |  CapRank = rank vs same sector + cap
  Week Vol = this week avg vs prior 15-day avg
  Break Vol= this week avg vs consolidation period avg
  ↑ = jumped 10%+ in rank vs last week

──────────────────────────────────────────────────────────────────────────────
  TIER 1 — MOMENTUM  (24 stocks)
  Fund ≥60 + Tech ≥65 + Momentum setup confirmed
──────────────────────────────────────────────────────────────────────────────

  [L] Large Cap
  ────────────────────────────────────────────────────────────────────

  #1  ABSLAMC  (Financial Services)  MCap Rs27,282Cr  Rs944.65
    Setup   : Momentum    Tech  65/100
    Scores  : Fund 73.0/100  Sector Rank 8.5/10  Cap Rank 8.5/10
    Tech    : RSI 52  ADX 22  MACD -2.47  → Momentum building, above EMA200, RSI 52
    Volume  : Week Vol:0.72x (Low) → Weak volume → mome